In [ ]:
"""
LLaMA 3 Implementation for LULC Event Extraction - Jupyter Notebook Version
===========================================================================

This notebook demonstrates how to use LLaMA 3 for extracting Land Use Land Cover (LULC) 
change events from text in a Jupyter notebook environment.

Requirements:
- Python 3.8+
- PyTorch 2.0+
- transformers 4.30.0+
- GPU with at least 16GB VRAM (for 8B model)
"""

# Cell 1: Import libraries and configure logging
import json
import os
import re
import logging
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from tqdm.notebook import tqdm

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.StreamHandler()
    ]
)

# Cell 2: Configuration settings
# You can modify these settings as needed
NER_OUTPUT_PATH = "label_studio_output/tasks.json"
OUTPUT_PATH = "llama3_extracted_lulc_events.csv"
MODEL_ID = "meta-llama/Meta-Llama-3-8B"  # 8B version
# Alternative models:
# "meta-llama/Meta-Llama-3-8B-Instruct" - Instruction-tuned version
# "meta-llama/Meta-Llama-3-70B" - Larger model (requires more VRAM)
NUM_SAMPLES = 0  # 0 for all samples, or specify a number to limit
USE_QUANTIZATION = True  # Set to True to use 4-bit quantization (reduces memory usage)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Cell 3: Setup model and tokenizer
def setup_model(model_id, device, use_quantization=False):
    """
    Load the LLaMA 3 model and tokenizer.
    
    Args:
        model_id: Hugging Face model ID
        device: Device to load the model on
        use_quantization: Whether to use 4-bit quantization
        
    Returns:
        model, tokenizer
    """
    logging.info(f"Loading LLaMA 3 tokenizer: {model_id}")
    
    # Load tokenizer
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    
    # Configure quantization if requested
    if use_quantization:
        logging.info("Using 4-bit quantization")
        quantization_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_use_double_quant=True
        )
    else:
        quantization_config = None
    
    # Load model with appropriate configuration
    logging.info(f"Loading LLaMA 3 model: {model_id}")
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        device_map=device,
        quantization_config=quantization_config,
        torch_dtype=torch.float16 if device == "cuda" else torch.float32,
    )
    
    logging.info(f"LLaMA 3 model loaded successfully on {device}")
    return model, tokenizer

# Execute this cell to load the model
model, tokenizer = setup_model(MODEL_ID, DEVICE, USE_QUANTIZATION)

# Cell 4: Prompt construction functions
def format_entities_for_prompt(entities):
    """Format entities for inclusion in the prompt."""
    if not entities:
        return "No specific entities pre-identified."
    
    # Group entities by type
    entities_by_type = {}
    for entity in entities:
        entity_type = entity.get('label', 'UNKNOWN')
        if entity_type not in entities_by_type:
            entities_by_type[entity_type] = []
        entities_by_type[entity_type].append(entity.get('text', 'N/A'))
    
    # Format grouped entities
    formatted_lines = []
    for entity_type, entity_texts in entities_by_type.items():
        unique_texts = list(set(entity_texts))  # Remove duplicates
        # Fixed line to avoid nested f-string syntax error
        entity_list = ", ".join(['"' + text + '"' for text in unique_texts])
        formatted_lines.append(f"- {entity_type}: {entity_list}")
    
    return "\n".join(formatted_lines)

def construct_llama3_prompt(sentence_text, entities):
    """
    Construct an optimized prompt for LLaMA 3.
    
    This prompt is specifically designed for LLaMA 3's capabilities and includes:
    1. Clear task definition
    2. Structured format instructions
    3. Few-shot examples
    4. Entity context
    """
    formatted_entities = format_entities_for_prompt(entities)
    
    # Extract LULC entities specifically for better context
    lulc_entities = [ent for ent in entities if ent.get('label') == 'LULC']
    # Fixed line to avoid nested f-string syntax error
    lulc_text = ", ".join(['"' + ent.get("text") + '"' for ent in lulc_entities]) if lulc_entities else "None identified"
    
    # Extract change indicators for better context
    change_entities = [ent for ent in entities if ent.get('label') == 'CHANGE']
    # Fixed line to avoid nested f-string syntax error
    change_text = ", ".join(['"' + ent.get("text") + '"' for ent in change_entities]) if change_entities else "None identified"
    
    prompt = f"""<task>
You are an expert in Land Use and Land Cover (LULC) analysis. Your task is to extract LULC change events from text.
</task>

<context>
LULC change events involve transitions or modifications in land types, often described with:
- Original land type (FROM)
- Resulting land type (TO)
- Words indicating change (CHANGE)
- Broader processes like deforestation or urbanization (PROCESS)
- Quantitative measurements (MAGNITUDE)
</context>

<input>
"{sentence_text}"
</input>

<entities>
{formatted_entities}
</entities>

<instructions>
Analyze the input text and extract a LULC change relation between the entities:
**Relationship Types - BE VERY THOUGHTFUL:**

**CHANGE_TO**: Indicates a direct transformation from one LULC type to another.
- ✅ CORRECT: forest --TRANSFORMS_TO-- cropland (trees cut, land converted to farming)
- ✅ CORRECT: agricultural land --TRANSFORMS_TO-- urban area (farmland developed into city)
- ✅ CORRECT: grassland --TRANSFORMS_TO-- built-up area (grass removed, buildings constructed)
- ❌ WRONG: built-up area --TRANSFORMS_TO-- built-up area (same type, just quantity change)
- ❌ WRONG: forest --TRANSFORMS_TO-- forest (same type, just area change)

**INCREASES_BY/DECREASES_BY**: For quantitative changes within same LULC type
- ✅ CORRECT: built-up area --INCREASES_BY-- 12.77% (more built-up area, not transformation)
- ✅ CORRECT: forest --DECREASES_BY-- 25% (less forest area, not transformation)

**Other Relations:**
- CAUSES: One entity causes a change (deforestation --CAUSES-- forest loss)
- LOCATED_IN: Spatial relationships (forest --LOCATED_IN-- Brazil)
- OCCURS_DURING: Temporal relationships (change --OCCURS_DURING-- 2018)
- MEASURES: Quantitative relationships (12.77% --MEASURES-- increase)
- AFFECTS: Impact relationships (urbanization --AFFECTS-- forest)
- FROM_TO: Value changes (52.88% --FROM_TO-- 65.5%)
- ENABLES: One process enables another (deforestation --ENABLES-- urbanization)
**Relationship Types - BE VERY THOUGHTFUL:**
...
- OCCURS_DURING: Temporal relationships where a **change, process** takes place or is observed within a specific time period. (e.g., change --OCCURS_DURING-- 2018, urbanization --OCCURS_DURING-- decade)
❌ WRONG: simulation results --OCCURS_DURING-- study period (results don't 'occur' in time, they are 'from' or 'valid for' a period)
**CRITICAL THINKING RULES:**
1. **Ask yourself**: Is this ACTUALLY a transformation between different land types?
2. **Think about the process**: What physical change happened to the land?
3. **Consider causality**: What caused what? Don't create meaningless loops
4. **Be precise with measurements**: Percentages usually MEASURE changes, not cause them
5. **Temporal logic**: Changes happen DURING time periods, not TO time periods
6. **Spatial logic**: Things are LOCATED_IN places, places don't transform to places
<output>
"""
    return prompt

# Cell 5: Text generation and parsing functions
def generate_with_llama3(model, tokenizer, prompt, max_new_tokens=256):
    """
    Generate text using LLaMA 3 model.
    
    Args:
        model: LLaMA 3 model
        tokenizer: LLaMA 3 tokenizer
        prompt: Input prompt
        max_new_tokens: Maximum number of tokens to generate
        
    Returns:
        Generated text
    """
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    
    # Generate with appropriate parameters for structured extraction
    with torch.no_grad():
        outputs = model.generate(
            inputs.input_ids,
            max_new_tokens=max_new_tokens,
            temperature=0.1,  # Low temperature for deterministic outputs
            top_p=0.9,
            do_sample=True,  # Light sampling for better quality
            num_return_sequences=1,
            pad_token_id=tokenizer.eos_token_id
        )
    
    # Decode and extract only the newly generated text
    full_output = tokenizer.decode(outputs[0], skip_special_tokens=True)
    generated_text = full_output[len(tokenizer.decode(inputs.input_ids[0], skip_special_tokens=True)):]
    
    # Clean up the output
    generated_text = generated_text.strip()
    
    # If the output contains </output> tag, extract only the content before it
    if "</output>" in generated_text:
        generated_text = generated_text.split("</output>")[0].strip()
    
    return generated_text

def parse_llama3_output(output_text):
    """
    Parse the LLaMA 3 output into structured fields.
    
    Args:
        output_text: Raw output from LLaMA 3
        
    Returns:
        Dictionary with parsed fields
    """
    # Check for NO_EVENT marker
    if "NO_EVENT" in output_text:
        return {
            "event_found": False,
            "from_lulc": "",
            "to_lulc": "",
            "change_indicator": "",
            "lulc_process": "",
            "magnitude": ""
        }
    
    # Extract fields using regex
    from_match = re.search(r'FROM:\s*(.*?)(?=\nTO:|$)', output_text, re.DOTALL)
    to_match = re.search(r'TO:\s*(.*?)(?=\nCHANGE:|$)', output_text, re.DOTALL)
    change_match = re.search(r'CHANGE:\s*(.*?)(?=\nPROCESS:|$)', output_text, re.DOTALL)
    process_match = re.search(r'PROCESS:\s*(.*?)(?=\nMAGNITUDE:|$)', output_text, re.DOTALL)
    magnitude_match = re.search(r'MAGNITUDE:\s*(.*?)(?=\n|$)', output_text, re.DOTALL)
    
    # Extract values or default to empty string
    from_lulc = from_match.group(1).strip() if from_match else ""
    to_lulc = to_match.group(1).strip() if to_match else ""
    change_indicator = change_match.group(1).strip() if change_match else ""
    lulc_process = process_match.group(1).strip() if process_match else ""
    magnitude = magnitude_match.group(1).strip() if magnitude_match else ""
    
    # Determine if an event was found (at least one field has content)
    event_found = bool(from_lulc or to_lulc or change_indicator or lulc_process)
    
    return {
        "event_found": event_found,
        "from_lulc": from_lulc,
        "to_lulc": to_lulc,
        "change_indicator": change_indicator,
        "lulc_process": lulc_process,
        "magnitude": magnitude
    }

def process_magnitude(magnitude):
    """
    Process magnitude into percent and area components.
    
    Args:
        magnitude: Raw magnitude string
        
    Returns:
        Tuple of (magnitude_percent, magnitude_area)
    """
    if not magnitude:
        return "", ""
    
    magnitude_percent = ""
    magnitude_area = ""
    
    # Check for percentage
    if "%" in magnitude:
        magnitude_percent = magnitude
    # Check for area units
    elif any(unit in magnitude.lower() for unit in ["ha", "km", "acre", "meter", "sq", "hectare"]):
        magnitude_area = magnitude
    # Check for numbers with area units using regex
    elif re.search(r'\d+\s*(?:ha|km|m|acre)', magnitude, re.IGNORECASE):
        magnitude_area = magnitude
    # If it's just a number, try to determine if it's a percentage
    elif re.search(r'\d+\.\d+|\d+', magnitude):
        try:
            value = float(re.search(r'\d+\.\d+|\d+', magnitude).group())
            if value <= 100:
                magnitude_percent = magnitude
            else:
                magnitude_area = magnitude
        except:
            magnitude_area = magnitude
    else:
        magnitude_area = magnitude
    
    return magnitude_percent, magnitude_area

# Cell 6: Load NER output data
# Execute this cell to load your data
try:
    logging.info(f"Loading NER output from {NER_OUTPUT_PATH}")
    with open(NER_OUTPUT_PATH, 'r', encoding='utf-8') as f:
        sentences_with_entities = json.load(f)
    
    total_sentences = len(sentences_with_entities)
    logging.info(f"Loaded {total_sentences} sentences with entities")
    
    # Limit number of samples if specified
    if NUM_SAMPLES > 0 and NUM_SAMPLES < total_sentences:
        sentences_with_entities = sentences_with_entities[:NUM_SAMPLES]
        logging.info(f"Processing first {NUM_SAMPLES} sentences")
    else:
        logging.info(f"Processing all {total_sentences} sentences")
        
    # Display first example
    print("\nFirst example:")
    print(f"Sentence: {sentences_with_entities[0].get('original_sentence', '')}")
    print("Entities:")
    for entity in sentences_with_entities[0].get('entities', []):
        print(f"  - {entity.get('text', '')} ({entity.get('label', '')})")
        
except Exception as e:
    logging.error(f"Error loading NER output: {e}")
    sentences_with_entities = []

# Cell 7: Process a single example (for testing)
# Execute this cell to test extraction on a single example
if sentences_with_entities:
    # Get the first example
    example = sentences_with_entities[0]
    sentence_text = example.get('original_sentence', '')
    entities = example.get('entities', [])
    
    # Construct prompt
    prompt = construct_llama3_prompt(sentence_text, entities)
    print("Prompt:")
    print(prompt)
    
    # Generate with LLaMA 3
    generated_text = generate_with_llama3(model, tokenizer, prompt)
    print("\nGenerated text:")
    print(generated_text)
    
    # Parse output
    parsed_result = parse_llama3_output(generated_text)
    print("\nParsed result:")
    for key, value in parsed_result.items():
        print(f"  {key}: {value}")
    
    # Process magnitude
    if parsed_result['magnitude']:
        magnitude_percent, magnitude_area = process_magnitude(parsed_result['magnitude'])
        print(f"  magnitude_percent: {magnitude_percent}")
        print(f"  magnitude_area: {magnitude_area}")

# Cell 8: Process all examples
# Execute this cell to process all examples
def process_all_examples():
    extracted_events = []
    
    logging.info("Starting LULC event extraction with LLaMA 3")
    for idx, entry in enumerate(tqdm(sentences_with_entities, desc="Processing sentences")):
        sentence_text = entry.get('original_sentence', '')
        entities = entry.get('entities', [])
        article_id = entry.get('article_id', f"UnknownID_{idx}")
        
        if not sentence_text.strip():
            logging.warning(f"Empty sentence for entry {idx}, skipping")
            continue
        
        # Prepare event data row
        event_data = {
            'article_id': article_id,
            'original_sentence': sentence_text,
            'llm_raw_output': 'Not Generated Yet',
            'event_found': False,
            'from_lulc': "",
            'to_lulc': "",
            'change_indicator': "",
            'lulc_process': "",
            'magnitude_percent': "",
            'magnitude_area': "",
            'error': None
        }
        
        try:
            # Construct prompt
            prompt = construct_llama3_prompt(sentence_text, entities)
            
            # Generate with LLaMA 3
            generated_text = generate_with_llama3(model, tokenizer, prompt)
            event_data['llm_raw_output'] = generated_text
            
            # Parse output
            parsed_result = parse_llama3_output(generated_text)
            
            # Update event data
            event_data['event_found'] = parsed_result['event_found']
            event_data['from_lulc'] = parsed_result['from_lulc']
            event_data['to_lulc'] = parsed_result['to_lulc']
            event_data['change_indicator'] = parsed_result['change_indicator']
            event_data['lulc_process'] = parsed_result['lulc_process']
            
            # Process magnitude
            magnitude_percent, magnitude_area = process_magnitude(parsed_result['magnitude'])
            event_data['magnitude_percent'] = magnitude_percent
            event_data['magnitude_area'] = magnitude_area
            
        except Exception as e:
            logging.error(f"Error processing entry {idx}: {e}")
            event_data['error'] = str(e)
            event_data['llm_raw_output'] = 'Error during generation'
        
        extracted_events.append(event_data)
    
    return extracted_events



2025-07-01 17:21:18,322 - INFO - Loading LLaMA 3 tokenizer: meta-llama/Meta-Llama-3-8B


tokenizer_config.json:   0%|          | 0.00/50.6k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/73.0 [00:00<?, ?B/s]

2025-07-01 17:21:20,617 - INFO - Using 4-bit quantization
2025-07-01 17:21:20,624 - INFO - Loading LLaMA 3 model: meta-llama/Meta-Llama-3-8B


config.json:   0%|          | 0.00/654 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`
2025-07-01 17:21:22,704 - WARNING - Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model-00001-of-00004.safetensors:   0%|          | 0.00/4.98G [00:00<?, ?B/s]

In [ ]:
def process_first_10_examples():
    extracted_events = []
    
    # Limit to first 10 examples
    examples_to_process = sentences_with_entities[:10]
    
    logging.info(f"Starting LULC event extraction with LLaMA 3 for first 10 examples")
    for idx, entry in enumerate(tqdm(examples_to_process, desc="Processing sentences")):
        sentence_text = entry.get('original_sentence', '')
        entities = entry.get('entities', [])
        article_id = entry.get('article_id', f"UnknownID_{idx}")
        
        if not sentence_text.strip():
            logging.warning(f"Empty sentence for entry {idx}, skipping")
            continue
        
        # Prepare event data row
        event_data = {
            'article_id': article_id,
            'original_sentence': sentence_text,
            'llm_raw_output': 'Not Generated Yet',
            'event_found': False,
            'from_lulc': "",
            'to_lulc': "",
            'change_indicator': "",
            'lulc_process': "",
            'magnitude_percent': "",
            'magnitude_area': "",
            'error': None
        }
        
        try:
            # Construct prompt
            prompt = construct_llama3_prompt(sentence_text, entities)
            
            # Generate with LLaMA 3
            generated_text = generate_with_llama3(model, tokenizer, prompt)
            event_data['llm_raw_output'] = generated_text
            
            # Parse output
            parsed_result = parse_llama3_output(generated_text)
            
            # Update event data
            event_data['event_found'] = parsed_result['event_found']
            event_data['from_lulc'] = parsed_result['from_lulc']
            event_data['to_lulc'] = parsed_result['to_lulc']
            event_data['change_indicator'] = parsed_result['change_indicator']
            event_data['lulc_process'] = parsed_result['lulc_process']
            
            # Process magnitude
            magnitude_percent, magnitude_area = process_magnitude(parsed_result['magnitude'])
            event_data['magnitude_percent'] = magnitude_percent
            event_data['magnitude_area'] = magnitude_area
            
        except Exception as e:
            logging.error(f"Error processing entry {idx}: {e}")
            event_data['error'] = str(e)
            event_data['llm_raw_output'] = 'Error during generation'
        
        extracted_events.append(event_data)
    
    return extracted_events

# Run this to process only the first 10 examples
extracted_events = process_first_10_examples()


In [ ]:
# Save first 10 results to a specific file
first_10_output_path = "llama3_examples.csv"
results_df = pd.DataFrame(extracted_events)
results_df.to_csv(first_10_output_path, index=False)
print(f"Saved first 10 results to {first_10_output_path}")

# Display the results
with pd.option_context('display.max_colwidth', 80, 'display.max_rows', 10, 'display.width', 1000):
    display(results_df[['article_id', 'event_found', 'from_lulc', 'to_lulc', 'change_indicator', 'lulc_process']])


In [ ]:
import pandas as pd
import json
import re
import logging
from typing import List, Dict, Any, Tuple, Optional
from IPython.display import display

# Configure logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

# Configuration
INPUT_CSV_PATH = "llama3_first_10_examples.csv"
OUTPUT_JSON_PATH = "labelstudio_relations.json"
RELATION_TYPES = ["causes", "affects", "located_in", "part_of", "temporal"]

# Helper: Find all spans of an entity in a sentence
def find_entity_spans(text: str, entity_text: str) -> List[Tuple[int, int]]:
    if not entity_text or not text:
        return []
    entity_regex = re.escape(entity_text.strip().lower())
    return [(m.start(), m.end()) for m in re.finditer(entity_regex, text.lower())]

# Helper: Determine relation type from change indicator and process
def determine_relation_type(from_lulc: str, to_lulc: str, change_indicator: str, lulc_process: str) -> str:
    if any(x in change_indicator.lower() for x in ["led to", "result in", "cause", "trigger", "induce"]):
        return "causes"
    if any(x in change_indicator.lower() for x in ["in", "at", "within", "near", "around"]):
        return "located_in"
    if any(x in change_indicator.lower() for x in ["part of", "component of", "element of"]):
        return "part_of"
    if any(x in change_indicator.lower() for x in ["before", "after", "during", "from", "to"]):
        return "temporal"
    if lulc_process and (from_lulc or to_lulc):
        return "causes"
    return "affects"

# Convert one row to Label Studio item
def create_label_studio_item(row: pd.Series, item_id: int) -> Optional[Dict[str, Any]]:
    text = row.get('original_sentence', '')
    from_lulc = row.get('from_lulc', '')
    to_lulc = row.get('to_lulc', '')
    change_indicator = row.get('change_indicator', '')
    lulc_process = row.get('lulc_process', '')
    if not text or not row.get('event_found', False):
        return None

    result = {
        "id": item_id,
        "data": {"text": text},
        "annotations": [{"id": f"{item_id}-annotation", "result": []}]
    }

    entities = []
    entity_id = 1

    def add_entity(label: str, entity_text: str, prefix: str):
        nonlocal entity_id
        spans = find_entity_spans(text, entity_text)
        for (start, end) in spans:
            entity = {
                "id": f"{prefix}-{entity_id}",
                "from_name": "label",
                "to_name": "text",
                "type": "labels",
                "value": {
                    "start": start,
                    "end": end,
                    "text": text[start:end],
                    "labels": [label]
                }
            }
            result["annotations"][0]["result"].append(entity)
            entities.append({"id": entity["id"], "type": label})
            entity_id += 1

    if from_lulc:
        add_entity("FROM_LULC", from_lulc, "from")
    if to_lulc:
        add_entity("TO_LULC", to_lulc, "to")
    if change_indicator:
        add_entity("CHANGE", change_indicator, "change")
    if lulc_process:
        add_entity("PROCESS", lulc_process, "process")

    relation_id = 1
    if len(entities) >= 2:
        relation_type = determine_relation_type(from_lulc, to_lulc, change_indicator, lulc_process)
        for i, source in enumerate(entities):
            for j, target in enumerate(entities):
                if i != j:
                    relation = {
                        "id": f"relation-{relation_id}",
                        "from_name": "relation",
                        "to_name": "text",
                        "type": "relation",
                        "value": {
                            "from_id": source["id"],
                            "to_id": target["id"],
                            "direction": "right",
                            "labels": [relation_type]
                        }
                    }
                    result["annotations"][0]["result"].append(relation)
                    relation_id += 1

    return result if entities else None

# Create the XML config for Label Studio
def create_label_studio_config(relation_types: List[str]) -> str:
    relation_labels = "\n".join([
        f'        <Relation value="{rel_type}" background="#{hash(rel_type) % 0xFFFFFF:06x}"/>'
        for rel_type in relation_types
    ])
    return f"""
<View>
  <Labels name="label" toName="text">
    <Label value="FROM_LULC" background="#8A2BE2"/>
    <Label value="TO_LULC" background="#FF6347"/>
    <Label value="CHANGE" background="#6495ED"/>
    <Label value="PROCESS" background="#32CD32"/>
  </Labels>
  <Relations name="relation" toName="text">
{relation_labels}
  </Relations>
  <Text name="text" value="$text"/>
</View>
"""

# Convert entire DataFrame
def convert_csv_to_labelstudio(df: pd.DataFrame, relation_types: List[str]) -> Dict[str, Any]:
    items = [create_label_studio_item(row, idx + 1) for idx, row in df.iterrows()]
    items = [item for item in items if item is not None]
    return {
        "config": create_label_studio_config(relation_types),
        "items": items
    }

# Save final JSON
def save_labelstudio_data(data: Dict[str, Any], output_path: str):
    with open(output_path, 'w', encoding='utf-8') as f:
        json.dump(data, f, indent=2, ensure_ascii=False)
    logging.info(f"Saved Label Studio JSON to {output_path}")

# Load CSV and convert
try:
    logging.info(f"Loading CSV from {INPUT_CSV_PATH}")
    df = pd.read_csv(INPUT_CSV_PATH)
    display(df.head())
    logging.info(f"{df['event_found'].sum()} events found in {len(df)} rows")
    labelstudio_data = convert_csv_to_labelstudio(df, RELATION_TYPES)
    save_labelstudio_data(labelstudio_data, OUTPUT_JSON_PATH)
except Exception as e:
    logging.error(f"Conversion failed: {e}")


In [ ]:
 custom_conversion("llama3_first_10_examples.csv", "labelstudio_relations.json")


In [ ]:
def process_all_examples():
    extracted_events = []
    
    # Limit to first 10 examples
    examples_to_process = sentences_with_entities
    
    logging.info(f"Starting LULC event extraction with LLaMA 3 for all examples")
    for idx, entry in enumerate(tqdm(examples_to_process, desc="Processing sentences")):
        sentence_text = entry.get('original_sentence', '')
        entities = entry.get('entities', [])
        article_id = entry.get('article_id', f"UnknownID_{idx}")
        
        if not sentence_text.strip():
            logging.warning(f"Empty sentence for entry {idx}, skipping")
            continue
        
        # Prepare event data row
        event_data = {
            'article_id': article_id,
            'original_sentence': sentence_text,
            'llm_raw_output': 'Not Generated Yet',
            'event_found': False,
            'from_lulc': "",
            'to_lulc': "",
            'change_indicator': "",
            'lulc_process': "",
            'magnitude_percent': "",
            'magnitude_area': "",
            'error': None
        }
        
        try:
            # Construct prompt
            prompt = construct_llama3_prompt(sentence_text, entities)
            
            # Generate with LLaMA 3
            generated_text = generate_with_llama3(model, tokenizer, prompt)
            event_data['llm_raw_output'] = generated_text
            
            # Parse output
            parsed_result = parse_llama3_output(generated_text)
            
            # Update event data
            event_data['event_found'] = parsed_result['event_found']
            event_data['from_lulc'] = parsed_result['from_lulc']
            event_data['to_lulc'] = parsed_result['to_lulc']
            event_data['change_indicator'] = parsed_result['change_indicator']
            event_data['lulc_process'] = parsed_result['lulc_process']
            
            # Process magnitude
            magnitude_percent, magnitude_area = process_magnitude(parsed_result['magnitude'])
            event_data['magnitude_percent'] = magnitude_percent
            event_data['magnitude_area'] = magnitude_area
            
        except Exception as e:
            logging.error(f"Error processing entry {idx}: {e}")
            event_data['error'] = str(e)
            event_data['llm_raw_output'] = 'Error during generation'
        
        extracted_events.append(event_data)
    
    return extracted_events

# Run this to process ALL examples
extracted_events = process_all_examples()


In [ ]:
import pandas as pd
import json
import re
import logging
from typing import List, Dict, Any, Tuple, Optional

# Configure logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

# Configuration
INPUT_CSV_PATH = "llama3_examples.csv"  # Example path
OUTPUT_JSON_PATH = "enhanced_labelstudio_tasks.json"

RELATION_TYPES = ["causes", "affects", "located_in", "part_of", "temporal", "converts_to"]

def safe_str(value) -> str:
    """Convert any value to string safely, handling None, NaN, and other types."""
    if pd.isna(value) or value is None:
        return ""
    return str(value).strip()

def find_entity_spans(text: str, entity_text: str) -> List[Tuple[int, int]]:
    """Find all occurrences of an entity in the text and return their spans."""
    text = safe_str(text)
    entity_text = safe_str(entity_text)
    
    if not entity_text or not text:
        return []
    
    try:
        entity_regex = re.escape(entity_text.lower())
        return [(m.start(), m.end()) for m in re.finditer(entity_regex, text.lower())]
    except Exception as e:
        logging.error(f"Error finding spans for '{entity_text}' in text: {e}")
        return []

def create_label_studio_item_with_enhanced_relations(row: pd.Series, item_id: int) -> Optional[Dict[str, Any]]:
    """Create a Label Studio compatible item with enhanced relations from a CSV row."""
    text = safe_str(row.get('original_sentence', ''))
    from_lulc = safe_str(row.get('from_lulc', ''))
    to_lulc = safe_str(row.get('to_lulc', ''))
    change_indicator = safe_str(row.get('change_indicator', ''))
    lulc_process = safe_str(row.get('lulc_process', ''))
    magnitude_percent = safe_str(row.get('magnitude_percent', ''))
    magnitude_area = safe_str(row.get('magnitude_area', ''))
    magnitude = magnitude_area if magnitude_area else magnitude_percent

    event_found = row.get('event_found', False)
    if isinstance(event_found, str):
        event_found = event_found.lower() == 'true'
    if not text or not event_found:
        return None

    result = {
        "id": item_id,
        "data": {
            "text": text
        },
        "annotations": [
            {
                "id": f"{item_id}-annotation",
                "result": []
            }
        ]
    }

    entities = []
    entity_id = 1
    def add_entity(entity_value, label_type, prefix):
        nonlocal entity_id
        spans = find_entity_spans(text, entity_value)
        added_ids = []
        for (start, end) in spans:
            eid = f"{prefix}-{entity_id}"
            result["annotations"][0]["result"].append({
                "id": eid,
                "from_name": "label",
                "to_name": "text",
                "type": "labels",
                "value": {
                    "start": start,
                    "end": end,
                    "text": text[start:end],
                    "labels": [label_type]
                }
            })
            entities.append({
                "id": eid,
                "type": prefix,
                "start": start,
                "end": end
            })
            added_ids.append(eid)
            entity_id += 1
        return added_ids[0] if added_ids else None

    from_entity_id = add_entity(from_lulc, "FROM_LULC", "from")
    to_entity_id = add_entity(to_lulc, "TO_LULC", "to")
    change_entity_id = add_entity(change_indicator, "CHANGE", "change")
    process_entity_id = add_entity(lulc_process, "PROCESS", "process")
    magnitude_entity_id = add_entity(magnitude, "MAGNITUDE", "magnitude")

    relation_id = 1
    def add_relation(from_id, to_id, label):
        nonlocal relation_id
        if from_id and to_id:
            result["annotations"][0]["result"].append({
                "id": f"relation-{relation_id}",
                "from_name": "relation",
                "to_name": "text",
                "type": "relation",
                "value": {
                    "from_id": from_id,
                    "to_id": to_id,
                    "direction": "right",
                    "labels": [label]
                }
            })
            relation_id += 1

    add_relation(from_entity_id, change_entity_id, "causes")
    add_relation(change_entity_id, process_entity_id, "part_of")
    add_relation(from_entity_id, to_entity_id, "converts_to")
    add_relation(change_entity_id, magnitude_entity_id, "affects")
    if not change_entity_id:
        add_relation(process_entity_id, magnitude_entity_id, "causes")

    return result if entities else None

def convert_csv_to_labelstudio_tasks(input_path, output_path):
    df = pd.read_csv(input_path)
    tasks = []
    for idx, row in df.iterrows():
        try:
            item = create_label_studio_item_with_enhanced_relations(row, idx + 1)
            if item:
                tasks.append(item)
        except Exception as e:
            logging.error(f"Error processing row {idx}: {e}")
    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(tasks, f, indent=2, ensure_ascii=False)
    logging.info(f"Exported {len(tasks)} tasks to {output_path}")
    return output_path

# Run the conversion
convert_csv_to_labelstudio_tasks(INPUT_CSV_PATH, OUTPUT_JSON_PATH)


In [ ]:
"""
Enhanced CSV to Label Studio Converter - Single Cell Version (Fixed)
===================================================================

This cell converts LULC event extraction results from CSV to Label Studio JSON
with enhanced relation creation between entities.
"""

import pandas as pd
import json
import re
import logging
from typing import List, Dict, Any, Tuple, Optional
from IPython.display import display, HTML

# Configure logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

# Configuration - MODIFY THESE VALUES
INPUT_CSV_PATH = "llama3_first_10_examples.csv"  # Path to your CSV file
OUTPUT_JSON_PATH = "enhanced_labelstudio_relations.json"  # Path to save the output JSON
RELATION_TYPES = ["causes", "affects", "located_in", "part_of", "temporal", "converts_to"]  # Relation types

def safe_str(value) -> str:
    """Convert any value to string safely, handling None, NaN, and other types."""
    if pd.isna(value) or value is None:
        return ""
    return str(value).strip()

def find_entity_spans(text: str, entity_text: str) -> List[Tuple[int, int]]:
    """Find all occurrences of an entity in the text and return their spans."""
    # Convert inputs to strings and handle None/NaN values
    text = safe_str(text)
    entity_text = safe_str(entity_text)
    
    if not entity_text or not text:
        return []
    
    try:
        # Clean entity text for regex search
        entity_regex = re.escape(entity_text.lower())
        
        # Find all occurrences (case insensitive)
        spans = []
        for match in re.finditer(entity_regex, text.lower()):
            spans.append((match.start(), match.end()))
        
        return spans
    except Exception as e:
        logging.error(f"Error finding spans for '{entity_text}' in text: {e}")
        return []

def create_label_studio_item_with_enhanced_relations(row: pd.Series, item_id: int) -> Optional[Dict[str, Any]]:
    """Create a Label Studio compatible item with enhanced relations from a CSV row."""
    # Extract relevant fields and convert to strings safely
    text = safe_str(row.get('original_sentence', ''))
    from_lulc = safe_str(row.get('from_lulc', ''))
    to_lulc = safe_str(row.get('to_lulc', ''))
    change_indicator = safe_str(row.get('change_indicator', ''))
    lulc_process = safe_str(row.get('lulc_process', ''))
    
    # Handle magnitude from either percent or area column
    magnitude_percent = safe_str(row.get('magnitude_percent', ''))
    magnitude_area = safe_str(row.get('magnitude_area', ''))
    magnitude = magnitude_area if magnitude_area else magnitude_percent
    
    # Check if event was found (convert to bool safely)
    event_found = row.get('event_found', False)
    if isinstance(event_found, str):
        event_found = event_found.lower() == 'true'
    
    # Skip if no text or no event found
    if not text or not event_found:
        return None
    
    # Initialize result structure
    result = {
        "id": item_id,
        "data": {
            "text": text
        },
        "annotations": [
            {
                "id": f"{item_id}-annotation",
                "result": []
            }
        ]
    }
    
    # Track entities and their IDs
    entities = []
    entity_id = 1
    
    # Process from_lulc entity
    from_entity_id = None
    if from_lulc:
        spans = find_entity_spans(text, from_lulc)
        if spans:
            for span_idx, (start, end) in enumerate(spans):
                from_entity_id = f"from-{entity_id}"
                entities.append({
                    "id": from_entity_id,
                    "type": "from_lulc",
                    "text": from_lulc,
                    "start": start,
                    "end": end
                })
                
                # Add to result
                result["annotations"][0]["result"].append({
                    "id": from_entity_id,
                    "from_name": "label",
                    "to_name": "text",
                    "type": "labels",
                    "value": {
                        "start": start,
                        "end": end,
                        "text": text[start:end],
                        "labels": ["FROM_LULC"]
                    }
                })
                entity_id += 1
    
    # Process to_lulc entity
    to_entity_id = None
    if to_lulc:
        spans = find_entity_spans(text, to_lulc)
        if spans:
            for span_idx, (start, end) in enumerate(spans):
                to_entity_id = f"to-{entity_id}"
                entities.append({
                    "id": to_entity_id,
                    "type": "to_lulc",
                    "text": to_lulc,
                    "start": start,
                    "end": end
                })
                
                # Add to result
                result["annotations"][0]["result"].append({
                    "id": to_entity_id,
                    "from_name": "label",
                    "to_name": "text",
                    "type": "labels",
                    "value": {
                        "start": start,
                        "end": end,
                        "text": text[start:end],
                        "labels": ["TO_LULC"]
                    }
                })
                entity_id += 1
    
    # Process change_indicator entity
    change_entity_id = None
    if change_indicator:
        spans = find_entity_spans(text, change_indicator)
        if spans:
            for span_idx, (start, end) in enumerate(spans):
                change_entity_id = f"change-{entity_id}"
                entities.append({
                    "id": change_entity_id,
                    "type": "change_indicator",
                    "text": change_indicator,
                    "start": start,
                    "end": end
                })
                
                # Add to result
                result["annotations"][0]["result"].append({
                    "id": change_entity_id,
                    "from_name": "label",
                    "to_name": "text",
                    "type": "labels",
                    "value": {
                        "start": start,
                        "end": end,
                        "text": text[start:end],
                        "labels": ["CHANGE"]
                    }
                })
                entity_id += 1
    
    # Process lulc_process entity
    process_entity_id = None
    if lulc_process:
        spans = find_entity_spans(text, lulc_process)
        if spans:
            for span_idx, (start, end) in enumerate(spans):
                process_entity_id = f"process-{entity_id}"
                entities.append({
                    "id": process_entity_id,
                    "type": "lulc_process",
                    "text": lulc_process,
                    "start": start,
                    "end": end
                })
                
                # Add to result
                result["annotations"][0]["result"].append({
                    "id": process_entity_id,
                    "from_name": "label",
                    "to_name": "text",
                    "type": "labels",
                    "value": {
                        "start": start,
                        "end": end,
                        "text": text[start:end],
                        "labels": ["PROCESS"]
                    }
                })
                entity_id += 1
    
    # Process magnitude entity
    magnitude_entity_id = None
    if magnitude:
        spans = find_entity_spans(text, magnitude)
        if spans:
            for span_idx, (start, end) in enumerate(spans):
                magnitude_entity_id = f"magnitude-{entity_id}"
                entities.append({
                    "id": magnitude_entity_id,
                    "type": "magnitude",
                    "text": magnitude,
                    "start": start,
                    "end": end
                })
                
                # Add to result
                result["annotations"][0]["result"].append({
                    "id": magnitude_entity_id,
                    "from_name": "label",
                    "to_name": "text",
                    "type": "labels",
                    "value": {
                        "start": start,
                        "end": end,
                        "text": text[start:end],
                        "labels": ["MAGNITUDE"]
                    }
                })
                entity_id += 1
    
    # Create enhanced relations between entities
    relation_id = 1
    
    # 1. FROM_LULC causes CHANGE (if both exist)
    if from_entity_id and change_entity_id:
        result["annotations"][0]["result"].append({
            "id": f"relation-{relation_id}",
            "from_name": "relation",
            "to_name": "text",
            "type": "relation",
            "value": {
                "from_id": from_entity_id,
                "to_id": change_entity_id,
                "direction": "right",
                "labels": ["causes"]
            }
        })
        relation_id += 1
    
    # 2. CHANGE part_of PROCESS (if both exist)
    if change_entity_id and process_entity_id:
        result["annotations"][0]["result"].append({
            "id": f"relation-{relation_id}",
            "from_name": "relation",
            "to_name": "text",
            "type": "relation",
            "value": {
                "from_id": change_entity_id,
                "to_id": process_entity_id,
                "direction": "right",
                "labels": ["part_of"]
            }
        })
        relation_id += 1
    
    # 3. FROM_LULC converts_to TO_LULC (if both exist)
    if from_entity_id and to_entity_id:
        result["annotations"][0]["result"].append({
            "id": f"relation-{relation_id}",
            "from_name": "relation",
            "to_name": "text",
            "type": "relation",
            "value": {
                "from_id": from_entity_id,
                "to_id": to_entity_id,
                "direction": "right",
                "labels": ["converts_to"]
            }
        })
        relation_id += 1
    
    # 4. CHANGE affects MAGNITUDE (if both exist)
    if change_entity_id and magnitude_entity_id:
        result["annotations"][0]["result"].append({
            "id": f"relation-{relation_id}",
            "from_name": "relation",
            "to_name": "text",
            "type": "relation",
            "value": {
                "from_id": change_entity_id,
                "to_id": magnitude_entity_id,
                "direction": "right",
                "labels": ["affects"]
            }
        })
        relation_id += 1
    
    # 5. PROCESS causes MAGNITUDE (if both exist and no CHANGE)
    if process_entity_id and magnitude_entity_id and not change_entity_id:
        result["annotations"][0]["result"].append({
            "id": f"relation-{relation_id}",
            "from_name": "relation",
            "to_name": "text",
            "type": "relation",
            "value": {
                "from_id": process_entity_id,
                "to_id": magnitude_entity_id,
                "direction": "right",
                "labels": ["causes"]
            }
        })
        relation_id += 1
    
    # If no entities were found, return None
    if not entities:
        return None
    
    return result

def create_label_studio_config(relation_types: List[str]) -> str:
    """Create Label Studio configuration XML for relation extraction."""
    relation_labels = "\n".join([f'        <Label value="{rel_type}" background="#{hash(rel_type) % 0xFFFFFF:06x}"/>' 
                                for rel_type in relation_types])
    
    config = f"""
    <View>
      <Relations name="relation" toName="text">
{relation_labels}
      </Relations>
      <Labels name="label" toName="text">
        <Label value="FROM_LULC" background="#8A2BE2"/>
        <Label value="TO_LULC" background="#FF6347"/>
        <Label value="CHANGE" background="#6495ED"/>
        <Label value="PROCESS" background="#32CD32"/>
        <Label value="MAGNITUDE" background="#FFD700"/>
      </Labels>
      <Text name="text" value="$text"/>
    </View>
    """
    
    return config

# Main conversion function
def convert_csv_to_enhanced_labelstudio(input_path, output_path, relation_types):
    """Convert CSV file to Label Studio JSON with enhanced relations."""
    try:
        # Load CSV file
        logging.info(f"Loading CSV file from {input_path}")
        df = pd.read_csv(input_path)
        logging.info(f"Loaded {len(df)} rows from CSV")
        
        # Display first few rows
        print("First few rows of the CSV:")
        display(df.head(3))
        
        # Create Label Studio items with enhanced relations
        items = []
        for idx, row in df.iterrows():
            try:
                item = create_label_studio_item_with_enhanced_relations(row, idx + 1)
                if item:
                    items.append(item)
            except Exception as e:
                logging.error(f"Error processing row {idx}: {e}")
                print(f"Error processing row {idx}: {e}")
                continue
        
        logging.info(f"Created {len(items)} Label Studio items with enhanced relations")
        
        # Create Label Studio config
        config = create_label_studio_config(relation_types)
        
        # Create final output
        output = {
            "config": config,
            "items": items
        }
        
        # Save to JSON file
        with open(output_path, 'w', encoding='utf-8') as f:
            json.dump(output, f, indent=2, ensure_ascii=False)
        
        logging.info(f"Saved enhanced Label Studio JSON to {output_path}")
        print(f"Successfully converted CSV to Label Studio JSON with enhanced relations")
        print(f"Output saved to: {output_path}")
        
        # Display sample of first item
        if items:
            print("\nSample of first item:")
            print(f"Text: {items[0]['data']['text']}")
            print(f"Entities: {len([r for r in items[0]['annotations'][0]['result'] if r['type'] == 'labels'])}")
            print(f"Relations: {len([r for r in items[0]['annotations'][0]['result'] if r['type'] == 'relation'])}")
            
            # Display relation types in first item
            relations = [r for r in items[0]['annotations'][0]['result'] if r['type'] == 'relation']
            if relations:
                print("\nRelation types in first item:")
                for rel in relations:
                    print(f"- {rel['value']['labels'][0]}")
        
        return output
    
    except Exception as e:
        logging.error(f"Error converting CSV to Label Studio JSON: {e}")
        print(f"Error: {e}")
        return None

# Execute the conversion
enhanced_labelstudio_data = convert_csv_to_enhanced_labelstudio(
    INPUT_CSV_PATH, 
    OUTPUT_JSON_PATH,
    RELATION_TYPES
)


In [ ]:
import pandas as pd
import json
import re
import logging
from typing import List, Tuple, Optional

# Configure logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

# Input/Output paths
INPUT_CSV_PATH = "llama3_first_10_examples.csv"
OUTPUT_JSON_PATH = "labelstudio_tasks_for_manual_annotation.json"

def safe_str(value) -> str:
    if pd.isna(value) or value is None:
        return ""
    return str(value).strip()

def clean_text_for_json(text):
    if not text:
        return ""
    text = text.replace('\n', ' ').replace('\r', ' ').replace('\t', ' ')
    return ''.join(ch for ch in text if ord(ch) >= 32 or ch in '\t\r\n')

def find_entity_spans(text: str, entity_text: str) -> List[Tuple[int, int]]:
    text = safe_str(text)
    entity_text = safe_str(entity_text)
    if not entity_text or not text:
        return []
    try:
        entity_regex = re.escape(entity_text.lower())
        return [(m.start(), m.end()) for m in re.finditer(entity_regex, text.lower())]
    except Exception as e:
        logging.error(f"Error finding spans for '{entity_text}' in text: {e}")
        return []

def create_label_studio_task(row: pd.Series, item_id: int) -> Optional[dict]:
    text = clean_text_for_json(safe_str(row.get('original_sentence', '')))
    event_found = row.get('event_found', False)
    if isinstance(event_found, str):
        event_found = event_found.lower() == 'true'
    if not text or not event_found:
        return None

    return {
        "id": item_id,
        "data": {
            "text": text
        }
    }

def convert_csv_to_labelstudio_tasks(input_path: str, output_path: str):
    df = pd.read_csv(input_path)
    tasks = []
    for idx, row in df.iterrows():
        task = create_label_studio_task(row, idx + 1)
        if task:
            tasks.append(task)

    with open(output_path, 'w', encoding='utf-8') as f:
        json.dump(tasks, f, indent=2, ensure_ascii=False)

    logging.info(f"Exported {len(tasks)} tasks to {output_path}")
    return output_path

# Run the conversion
convert_csv_to_labelstudio_tasks(INPUT_CSV_PATH, OUTPUT_JSON_PATH)


In [ ]:
import pandas as pd
import json
import re
import logging
from typing import List, Tuple, Optional

# Configure logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

# Input/Output paths
INPUT_CSV_PATH = "llama3_first_10_examples.csv"
OUTPUT_JSON_PATH = "labelstudio_annotated_with_relations.json"

def safe_str(value) -> str:
    if pd.isna(value) or value is None:
        return ""
    return str(value).strip()

def clean_text_for_json(text):
    if not text:
        return ""
    text = text.replace('\n', ' ').replace('\r', ' ').replace('\t', ' ')
    return ''.join(ch for ch in text if ord(ch) >= 32 or ch in '\t\r\n')

def find_entity_spans(text: str, entity_text: str) -> List[Tuple[int, int]]:
    text = safe_str(text)
    entity_text = safe_str(entity_text)
    if not entity_text or not text:
        return []
    try:
        entity_regex = re.escape(entity_text.lower())
        return [(m.start(), m.end()) for m in re.finditer(entity_regex, text.lower())]
    except Exception as e:
        logging.error(f"Error finding spans for '{entity_text}' in text: {e}")
        return []

def create_annotated_task(row: pd.Series, item_id: int) -> Optional[dict]:
    text = clean_text_for_json(safe_str(row.get('original_sentence', '')))
    from_lulc = clean_text_for_json(safe_str(row.get('from_lulc', '')))
    to_lulc = clean_text_for_json(safe_str(row.get('to_lulc', '')))
    change = clean_text_for_json(safe_str(row.get('change_indicator', '')))
    process = clean_text_for_json(safe_str(row.get('lulc_process', '')))
    magnitude = clean_text_for_json(safe_str(row.get('magnitude_area', '')) or safe_str(row.get('magnitude_percent', '')))
    event_found = row.get('event_found', False)

    if isinstance(event_found, str):
        event_found = event_found.lower() == 'true'
    if not text or not event_found:
        return None

    annotation = {
        "id": f"{item_id}-annotation",
        "result": []
    }

    result = {
        "id": item_id,
        "data": {
            "text": text
        },
        "annotations": [annotation]
    }

    entity_id = 1
    relations = []
    id_map = {}

    def add_entity(label_text, label_type, prefix):
        nonlocal entity_id
        spans = find_entity_spans(text, label_text)
        for (start, end) in spans:
            eid = f"{prefix}-{entity_id}"
            annotation["result"].append({
                "id": eid,
                "from_name": "label",
                "to_name": "text",
                "type": "labels",
                "value": {
                    "start": start,
                    "end": end,
                    "text": text[start:end],
                    "labels": [label_type]
                }
            })
            id_map[prefix] = eid
            entity_id += 1
            break  # only take the first match

    add_entity(from_lulc, "FROM_LULC", "from")
    add_entity(to_lulc, "TO_LULC", "to")
    add_entity(change, "CHANGE", "change")
    add_entity(process, "PROCESS", "process")
    add_entity(magnitude, "MAGNITUDE", "magnitude")

    def add_relation(from_key, to_key, label):
        if from_key in id_map and to_key in id_map:
            annotation["result"].append({
                "id": f"relation-{len(annotation['result'])+1}",
                "from_name": "relation",
                "to_name": "text",
                "type": "relation",
                "value": {
                    "from_id": id_map[from_key],
                    "to_id": id_map[to_key],
                    "direction": "right",
                    "labels": [label]
                }
            })

    add_relation("from", "change", "causes")
    add_relation("change", "process", "part_of")
    add_relation("from", "to", "converts_to")
    add_relation("change", "magnitude", "affects")
    if "process" in id_map and "magnitude" in id_map and "change" not in id_map:
        add_relation("process", "magnitude", "causes")

    return result if annotation["result"] else None

def convert_csv_to_annotated_tasks(input_path: str, output_path: str):
    df = pd.read_csv(input_path)
    tasks = []
    for idx, row in df.iterrows():
        task = create_annotated_task(row, idx + 1)
        if task:
            tasks.append(task)

    with open(output_path, 'w', encoding='utf-8') as f:
        json.dump(tasks, f, indent=2, ensure_ascii=False)

    logging.info(f"Exported {len(tasks)} tasks to {output_path}")
    return output_path

# Run the conversion
convert_csv_to_annotated_tasks(INPUT_CSV_PATH, OUTPUT_JSON_PATH)


In [ ]:
import pandas as pd
import json
import re
import logging
from typing import List, Dict, Any, Tuple, Optional
from IPython.display import display, HTML

# Configure logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

# Configuration - MODIFY THESE VALUES
INPUT_CSV_PATH = "llama3_first_10_examples.csv"  # Path to your CSV file
OUTPUT_JSON_PATH = "labelstudio_tasks_op1_notebook.json"  # Path to save the output JSON

def safe_str(value) -> str:
    """Convert any value to string safely, handling None, NaN, and other types."""
    if pd.isna(value) or value is None:
        return ""
    return str(value).strip()

def clean_text_for_json(text):
    """Clean text to ensure it's valid in JSON."""
    if not text:
        return ""
    
    text = text.replace('\n', ' ')
    text = text.replace('\r', ' ')
    text = text.replace('\t', ' ')
    text = ''.join(ch for ch in text if ord(ch) >= 32 or ch in '\t\r\n')
    return text

def find_entity_spans(text: str, entity_text: str) -> List[Tuple[int, int]]:
    """Find all occurrences of an entity in the text and return their spans."""
    text = safe_str(text)
    entity_text = safe_str(entity_text)
    
    if not entity_text or not text:
        return []
    
    try:
        entity_regex = re.escape(entity_text.lower())
        spans = []
        for match in re.finditer(entity_regex, text.lower()):
            spans.append((match.start(), match.end()))
        return spans
    except Exception as e:
        logging.error(f"Error finding spans for '{entity_text}' in text: {e}")
        return []

def create_label_studio_item(row: pd.Series, item_id_counter: int) -> Optional[Dict[str, Any]]:
    """Create a Label Studio compatible item from a CSV row."""
    text = clean_text_for_json(safe_str(row.get('original_sentence', '')))
    from_lulc_text = clean_text_for_json(safe_str(row.get('from_lulc', '')))
    to_lulc_text = clean_text_for_json(safe_str(row.get('to_lulc', '')))
    change_indicator_text = clean_text_for_json(safe_str(row.get('change_indicator', '')))
    lulc_process_text = clean_text_for_json(safe_str(row.get('lulc_process', '')))
    
    magnitude_percent = clean_text_for_json(safe_str(row.get('magnitude_percent', '')))
    magnitude_area = clean_text_for_json(safe_str(row.get('magnitude_area', '')))
    magnitude_text = magnitude_area if magnitude_area else magnitude_percent
    
    event_found = row.get('event_found', False)
    if isinstance(event_found, str):
        event_found = event_found.lower() == 'true'
    
    if not text or not event_found:
        return None
    
    result = {
        "data": {"text": text},
        "annotations": [{"result": []}] 
    }
    annotation_results = result["annotations"][0]["result"]
    
    # This unique ID counter is for labels and relations *within* a single task item.
    # The item_id_counter passed to the function is for the overall task ID.
    current_annotation_element_id = 1 

    first_from_lulc_id_for_relation = None
    first_to_lulc_id_for_relation = None
    first_change_indicator_id_for_relation = None
    first_lulc_process_id_for_relation = None
    first_magnitude_id_for_relation = None
    
    # --- Process FROM_LULC entity ---
    if from_lulc_text:
        spans = find_entity_spans(text, from_lulc_text)
        if spans:
            for span_idx, (start, end) in enumerate(spans):
                # Generate a unique ID for this label instance within the task
                label_instance_id = f"lbl-{current_annotation_element_id}"
                if span_idx == 0: 
                    first_from_lulc_id_for_relation = label_instance_id
                
                annotation_results.append({
                    "id": label_instance_id, "from_name": "label", "to_name": "text", "type": "labels",
                    "value": {"start": start, "end": end, "text": text[start:end], "labels": ["FROM_LULC"]}
                })
                current_annotation_element_id += 1
    
    # --- Process TO_LULC entity ---
    if to_lulc_text:
        spans = find_entity_spans(text, to_lulc_text)
        if spans:
            for span_idx, (start, end) in enumerate(spans):
                label_instance_id = f"lbl-{current_annotation_element_id}"
                if span_idx == 0:
                    first_to_lulc_id_for_relation = label_instance_id
                
                annotation_results.append({
                    "id": label_instance_id, "from_name": "label", "to_name": "text", "type": "labels",
                    "value": {"start": start, "end": end, "text": text[start:end], "labels": ["TO_LULC"]}
                })
                current_annotation_element_id += 1

    # --- Process CHANGE_INDICATOR entity ---
    if change_indicator_text:
        spans = find_entity_spans(text, change_indicator_text)
        if spans:
            for span_idx, (start, end) in enumerate(spans):
                label_instance_id = f"lbl-{current_annotation_element_id}"
                if span_idx == 0:
                    first_change_indicator_id_for_relation = label_instance_id
                
                annotation_results.append({
                    "id": label_instance_id, "from_name": "label", "to_name": "text", "type": "labels",
                    "value": {"start": start, "end": end, "text": text[start:end], "labels": ["CHANGE"]}
                })
                current_annotation_element_id += 1

    # --- Process LULC_PROCESS entity ---
    if lulc_process_text:
        spans = find_entity_spans(text, lulc_process_text)
        if spans:
            for span_idx, (start, end) in enumerate(spans):
                label_instance_id = f"lbl-{current_annotation_element_id}"
                if span_idx == 0:
                    first_lulc_process_id_for_relation = label_instance_id
                
                annotation_results.append({
                    "id": label_instance_id, "from_name": "label", "to_name": "text", "type": "labels",
                    "value": {"start": start, "end": end, "text": text[start:end], "labels": ["PROCESS"]}
                })
                current_annotation_element_id += 1

    # --- Process MAGNITUDE entity ---
    if magnitude_text:
        spans = find_entity_spans(text, magnitude_text)
        if spans:
            for span_idx, (start, end) in enumerate(spans):
                label_instance_id = f"lbl-{current_annotation_element_id}"
                if span_idx == 0:
                    first_magnitude_id_for_relation = label_instance_id
                
                annotation_results.append({
                    "id": label_instance_id, "from_name": "label", "to_name": "text", "type": "labels",
                    "value": {"start": start, "end": end, "text": text[start:end], "labels": ["MAGNITUDE"]}
                })
                current_annotation_element_id += 1
    
    # --- Create relations using the FIRST occurrence IDs ---
    
    # 1. FROM_LULC causes CHANGE
    if first_from_lulc_id_for_relation and first_change_indicator_id_for_relation:
        annotation_results.append({
            "id": f"rel-{current_annotation_element_id}", "from_name": "relation", "to_name": "text", "type": "relation",
            "value": {
                "from_id": first_from_lulc_id_for_relation, "to_id": first_change_indicator_id_for_relation,
                "direction": "right", "labels": ["causes"]
            }
        })
        current_annotation_element_id += 1
    
    # 2. CHANGE part_of PROCESS
    if first_change_indicator_id_for_relation and first_lulc_process_id_for_relation:
        annotation_results.append({
            "id": f"rel-{current_annotation_element_id}", "from_name": "relation", "to_name": "text", "type": "relation",
            "value": {
                "from_id": first_change_indicator_id_for_relation, "to_id": first_lulc_process_id_for_relation,
                "direction": "right", "labels": ["part_of"]
            }
        })
        current_annotation_element_id += 1
    
    # 3. FROM_LULC converts_to TO_LULC
    if first_from_lulc_id_for_relation and first_to_lulc_id_for_relation:
        annotation_results.append({
            "id": f"rel-{current_annotation_element_id}", "from_name": "relation", "to_name": "text", "type": "relation",
            "value": {
                "from_id": first_from_lulc_id_for_relation, "to_id": first_to_lulc_id_for_relation,
                "direction": "right", "labels": ["converts_to"]
            }
        })
        current_annotation_element_id += 1
    
    # 4. CHANGE affects MAGNITUDE
    if first_change_indicator_id_for_relation and first_magnitude_id_for_relation:
        annotation_results.append({
            "id": f"rel-{current_annotation_element_id}", "from_name": "relation", "to_name": "text", "type": "relation",
            "value": {
                "from_id": first_change_indicator_id_for_relation, "to_id": first_magnitude_id_for_relation,
                "direction": "right", "labels": ["affects"]
            }
        })
        current_annotation_element_id += 1
    
    if not annotation_results: 
        return None
    
    return result

# Main conversion function
def convert_csv_to_labelstudio(input_path, output_path):
    """Convert CSV file to Label Studio JSON."""
    try:
        logging.info(f"Loading CSV file from {input_path}")
        df = pd.read_csv(input_path)
        logging.info(f"Loaded {len(df)} rows from CSV")
        
        print("First few rows of the CSV:")
        display(df.head(3)) # display is fine in Jupyter notebooks
        
        tasks = []
        for idx, row in df.iterrows(): # idx will be the original DataFrame index
            try:
                # Pass idx + 1 as a unique ID for the Label Studio task item itself
                item = create_label_studio_item(row, idx + 1) 
                if item:
                    item["id"] = idx + 1 # Assign overall task ID
                    tasks.append(item)
            except Exception as e:
                logging.error(f"Error processing row {idx} (Task ID {idx+1}): {e}", exc_info=True)
                print(f"Error processing row {idx} (Task ID {idx+1}): {e}")
                continue
        
        logging.info(f"Created {len(tasks)} Label Studio tasks")
        
        with open(output_path, 'w', encoding='utf-8') as f:
            json.dump(tasks, f, ensure_ascii=False, indent=2)
        
        logging.info(f"Saved Label Studio JSON to {output_path}")
        print(f"Successfully converted CSV to Label Studio JSON")
        print(f"Output saved to: {output_path}")
        
        try:
            with open(output_path, 'r', encoding='utf-8') as f:
                json.load(f)
            print("JSON validation successful - file is valid JSON")
        except json.JSONDecodeError as e:
            print(f"WARNING: Generated JSON is invalid: {e}")
            print("Please check the output file manually")
        
        return tasks
    
    except Exception as e:
        logging.error(f"Error converting CSV to Label Studio JSON: {e}", exc_info=True)
        print(f"Error: {e}")
        return None

# Execute the conversion
labelstudio_tasks = convert_csv_to_labelstudio(INPUT_CSV_PATH, OUTPUT_JSON_PATH)

# Display Label Studio configuration instructions
if labelstudio_tasks is not None: 
    print("\n=== Label Studio Configuration ===")
    print("When creating your project in Label Studio, use this configuration:")
    print("""
<View>
  <Text name="text" value="$text"/>

  <Labels name="label" toName="text">
    <Label value="FROM_LULC" background="#8A2BE2"/>
    <Label value="TO_LULC" background="#FF6347"/>
    <Label value="CHANGE" background="#6495ED"/>
    <Label value="PROCESS" background="#32CD32"/>
    <Label value="MAGNITUDE" background="#FFD700"/>
  </Labels>

  <Relations name="relation" toName="text">
    <Relation value="causes" background="#ff9900"/>
    <Relation value="affects" background="#5cdaed"/>
    <Relation value="located_in" background="#ed5c8c"/> <!-- Note: located_in is in your config but not used by the script -->
    <Relation value="part_of" background="#5ced71"/>
    <Relation value="converts_to" background="#bf5ced"/>
  </Relations>
</View>
""")

In [ ]:
import pandas as pd
import json
import uuid

# Load the CSV-like data from the string into a DataFrame
csv_data = """
article_id,original_sentence,llm_raw_output,event_found,from_lulc,to_lulc,change_indicator,lulc_process,magnitude_percent,magnitude_area,error
Article_1,"Simulation results reveal that the landscape of Thimphu city has changed considerably during the study period and the change trend is predicted to continue into 2050.","FROM: \nTO: \nCHANGE: \nPROCESS: \nMAGNITUDE:",True,"TO: \nCHANGE: \nPROCESS: \nMAGNITUDE:","CHANGE: \nPROCESS: \nMAGNITUDE:","PROCESS: \nMAGNITUDE:",MAGNITUDE:,,,
Article_1,"The study observed a significant increase (12.77%) in built-up area from 2002 (52.88%) to 2018 (65.5%), followed by a slight increase in the cover of bare ground.","FROM: built-up area\nTO: built-up area\nCHANGE: increased\nPROCESS: urbanization\nMAGNITUDE: 12.77%",True,built-up area,built-up area,increased,urbanization,12.77%,,
Article_1,"On the contrary, forest cover declined drastically (15.25%) followed by agriculture (1.01%).","FROM: \nTO: \nCHANGE: \nPROCESS: \nMAGNITUDE:",True,"TO: \nCHANGE: \nPROCESS: \nMAGNITUDE:","CHANGE: \nPROCESS: \nMAGNITUDE:","PROCESS: \nMAGNITUDE:",MAGNITUDE:,,,
Article_1,"Rapid population growth triggered by rural urban migration coupled with hasty socio-economic development post democracy are the main drivers of these changes.","FROM: agricultural land\nTO: urban areas\nCHANGE: converted\nPROCESS: urbanization\nMAGNITUDE:",True,agricultural land,urban areas,converted,urbanization,,,
Article_1,"Under the business as usual scenario, prediction analysis for the year 2050 show that built up area will consume almost all of the city area (73.21%) with forest significantly reduced to patches making up only about 16% of the city.","FROM: built-up area\nTO: built-up area\nCHANGE: increased\nPROCESS: urbanization\nMAGNITUDE: 12.77%",True,built-up area,built-up area,increased,urbanization,12.77%,,
"""

# Read the CSV string into a pandas DataFrame
from io import StringIO
df = pd.read_csv(StringIO(csv_data))

# Create pre-annotation format
tasks = []
for idx, row in df.iterrows():
    if not row["event_found"]:
        continue

    sentence = row["original_sentence"]
    from_lulc = str(row["from_lulc"])
    to_lulc = str(row["to_lulc"])

    results = []
    entities = {}

    # Add from_lulc entity
    if from_lulc and isinstance(from_lulc, str) and from_lulc.strip() in sentence:
        start = sentence.index(from_lulc)
        end = start + len(from_lulc)
        id1 = str(uuid.uuid4())
        entities["from"] = id1
        results.append({
            "id": id1,
            "type": "labels",
            "value": {"start": start, "end": end, "text": from_lulc, "labels": ["FROM_LULC"]}
        })

    # Add to_lulc entity
    if to_lulc and isinstance(to_lulc, str) and to_lulc.strip() in sentence:
        start = sentence.index(to_lulc)
        end = start + len(to_lulc)
        id2 = str(uuid.uuid4())
        entities["to"] = id2
        results.append({
            "id": id2,
            "type": "labels",
            "value": {"start": start, "end": end, "text": to_lulc, "labels": ["TO_LULC"]}
        })

    # Add relation
    if "from" in entities and "to" in entities:
        results.append({
            "type": "relation",
            "from_id": entities["from"],
            "to_id": entities["to"],
            "labels": ["CHANGE_EVENT"]
        })

    if results:
        tasks.append({
            "data": {"text": sentence},
            "annotations": [{"result": results}]
        })

# Save as JSON
output_path = "labelstudio_preannotated.json"
with open(output_path, "w") as f:
    json.dump(tasks, f, indent=2)

output_path


In [ ]:
import json
import uuid
import pandas as pd
from typing import List, Dict
from io import StringIO

# Simulated CSV data (can be replaced with file read)
csv_data = """
article_id,original_sentence,llm_raw_output,event_found,from_lulc,to_lulc,change_indicator,lulc_process,magnitude_percent,magnitude_area,error
Article_1,"The study observed a significant increase (12.77%) in built-up area from 2002 (52.88%) to 2018 (65.5%), followed by a slight increase in the cover of bare ground.","FROM: built-up area\nTO: built-up area\nCHANGE: increased\nPROCESS: urbanization\nMAGNITUDE: 12.77%",True,built-up area,built-up area,increased,urbanization,12.77%,,
"""

df = pd.read_csv(StringIO(csv_data))

# Output tasks list
tasks: List[Dict] = []

# Iterate through rows to build the Label Studio pre-annotation format
for idx, row in df.iterrows():
    if not row["event_found"]:
        continue

    text = row["original_sentence"]
    from_lulc = str(row["from_lulc"]).strip()
    to_lulc = str(row["to_lulc"]).strip()

    results = []
    entity_ids = {}

    # FROM_LULC entity
    if from_lulc and from_lulc in text:
        start = text.index(from_lulc)
        end = start + len(from_lulc)
        id1 = str(uuid.uuid4())
        entity_ids["from"] = id1
        results.append({
            "id": id1,
            "from_name": "label",
            "to_name": "text",
            "type": "labels",
            "value": {"start": start, "end": end, "text": from_lulc, "labels": ["FROM_LULC"]}
        })

    # TO_LULC entity
    if to_lulc and to_lulc in text:
        start = text.index(to_lulc)
        end = start + len(to_lulc)
        id2 = str(uuid.uuid4())
        entity_ids["to"] = id2
        results.append({
            "id": id2,
            "from_name": "label",
            "to_name": "text",
            "type": "labels",
            "value": {"start": start, "end": end, "text": to_lulc, "labels": ["TO_LULC"]}
        })

    # Add relation if both entities are present
    if "from" in entity_ids and "to" in entity_ids:
        results.append({
            "type": "relation",
            "from_id": entity_ids["from"],
            "to_id": entity_ids["to"],
            "labels": ["CHANGE_EVENT"],
            "direction": "right"
        })

    if results:
        task = {
            "id": idx + 1,
            "data": {"text": text},
            "annotations": [{"id": idx + 1000, "result": results}]
        }
        tasks.append(task)

# Save to JSON
output_json_path = "labelstudio_preannotated.json"
with open(output_json_path, "w") as f:
    json.dump(tasks, f, indent=2)

output_json_path


In [ ]:
import uuid
from pathlib import Path

# Simulate extracted events from the LLaMA 3 pipeline
extracted_events = [
    {
        'article_id': 'Article_1',
        'original_sentence': 'The study observed a significant increase (12.77%) in built-up area from 2002 (52.88%) to 2018 (65.5%), followed by a slight increase in the cover of bare ground.',
        'llm_raw_output': 'FROM: built-up area\nTO: built-up area\nCHANGE: increased\nPROCESS: urbanization\nMAGNITUDE: 12.77%',
        'event_found': True,
        'from_lulc': 'built-up area',
        'to_lulc': 'built-up area',
        'change_indicator': 'increased',
        'lulc_process': 'urbanization',
        'magnitude_percent': '12.77%',
        'magnitude_area': '',
        'error': None
    }
]

# Create the Label Studio JSON format
labelstudio_tasks = []
for idx, row in enumerate(extracted_events):
    if not row["event_found"]:
        continue

    sentence = row["original_sentence"]
    from_lulc = row["from_lulc"]
    to_lulc = row["to_lulc"]

    results = []
    ids = {}

    # FROM_LULC entity
    if from_lulc in sentence:
        start = sentence.index(from_lulc)
        end = start + len(from_lulc)
        eid = str(uuid.uuid4())
        ids["from"] = eid
        results.append({
            "id": eid,
            "from_name": "label",
            "to_name": "text",
            "type": "labels",
            "value": {"start": start, "end": end, "text": from_lulc, "labels": ["FROM_LULC"]}
        })

    # TO_LULC entity
    if to_lulc in sentence:
        start = sentence.index(to_lulc)
        end = start + len(to_lulc)
        eid = str(uuid.uuid4())
        ids["to"] = eid
        results.append({
            "id": eid,
            "from_name": "label",
            "to_name": "text",
            "type": "labels",
            "value": {"start": start, "end": end, "text": to_lulc, "labels": ["TO_LULC"]}
        })

    # Relation
    if "from" in ids and "to" in ids:
        results.append({
            "type": "relation",
            "from_id": ids["from"],
            "to_id": ids["to"],
            "labels": ["CHANGE_EVENT"],
            "direction": "right"
        })

    if results:
        labelstudio_tasks.append({
            "id": idx + 1,
            "data": {"text": sentence},
            "annotations": [{"id": idx + 1000, "result": results}]
        })

# Save to a JSON file
output_json_path = Path("llama3_labelstudio_events.json")
with open(output_json_path, "w", encoding="utf-8") as f:
    json.dump(labelstudio_tasks, f, indent=2)

str(output_json_path)


In [ ]:
import pandas as pd
import json
import uuid
from pathlib import Path

# Load your full CSV file (replace with actual path if needed)
csv_path = "llama3_first_10_examples.csv"
df = pd.read_csv(csv_path)

# Define function to find the span of a text within a sentence
def find_span(text, sentence):
    try:
        start = sentence.index(text)
        end = start + len(text)
        return start, end
    except ValueError:
        return None, None

# Convert DataFrame to Label Studio JSON format
labelstudio_data = []
for idx, row in df.iterrows():
    sentence = str(row["original_sentence"])
    from_lulc = str(row.get("from_lulc", "")).strip()
    to_lulc = str(row.get("to_lulc", "")).strip()
    change = str(row.get("change_indicator", "")).strip()
    process = str(row.get("lulc_process", "")).strip()
    magnitude_percent = str(row.get("magnitude_percent", "")).strip()
    magnitude_area = str(row.get("magnitude_area", "")).strip()

    entities = {}
    results = []

    def add_entity(label, text):
        if text and text.lower() != "nan" and text in sentence:
            start, end = find_span(text, sentence)
            if start is not None:
                eid = str(uuid.uuid4())
                results.append({
                    "id": eid,
                    "from_name": "label",
                    "to_name": "text",
                    "type": "labels",
                    "value": {"start": start, "end": end, "text": text, "labels": [label]}
                })
                return eid
        return None

    # Add all entities
    entities['from'] = add_entity("FROM_LULC", from_lulc)
    entities['to'] = add_entity("TO_LULC", to_lulc)
    entities['change'] = add_entity("CHANGE", change)
    entities['process'] = add_entity("PROCESS", process)
    entities['magnitude'] = add_entity("MAGNITUDE", magnitude_percent or magnitude_area)

    # Define relations based on available entities
    if entities['from'] and entities['to']:
        results.append({
            "type": "relation",
            "from_id": entities['from'],
            "to_id": entities['to'],
            "labels": ["located_in"],
            "direction": "right"
        })
    if entities['change'] and entities['process']:
        results.append({
            "type": "relation",
            "from_id": entities['change'],
            "to_id": entities['process'],
            "labels": ["causes"],
            "direction": "right"
        })
    if entities['process'] and entities['to']:
        results.append({
            "type": "relation",
            "from_id": entities['process'],
            "to_id": entities['to'],
            "labels": ["affects"],
            "direction": "right"
        })
    if entities['from'] and entities['change']:
        results.append({
            "type": "relation",
            "from_id": entities['from'],
            "to_id": entities['change'],
            "labels": ["temporal"],
            "direction": "right"
        })

    if results:
        labelstudio_data.append({
            "id": idx + 1,
            "data": {"text": sentence},
            "annotations": [{"id": idx + 1000, "result": results}]
        })

# Save the final JSON
output_path = Path("llama3_full_labelstudio.json")
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(labelstudio_data, f, indent=2)

str(output_path)


In [ ]:
import pandas as pd
import json
import uuid

# Load the full CSV
df = pd.read_csv("llama3_first_10_examples.csv")

labelstudio_data = []

def get_span(text, sentence):
    try:
        start = sentence.index(text)
        return start, start + len(text)
    except ValueError:
        return None, None

for idx, row in df.iterrows():
    sentence = str(row["original_sentence"])
    results = []
    entity_ids = {}

    def add_entity(label, content):
        if content and content.lower() != "nan" and content in sentence:
            start, end = get_span(content, sentence)
            if start is not None:
                eid = str(uuid.uuid4())
                results.append({
                    "id": eid,
                    "from_name": "label",
                    "to_name": "text",
                    "type": "labels",
                    "value": {"start": start, "end": end, "text": content, "labels": [label]}
                })
                return eid
        return None

    # Add all entity labels
    entity_ids["FROM_LULC"] = add_entity("FROM_LULC", str(row.get("from_lulc", "")).strip())
    entity_ids["TO_LULC"] = add_entity("TO_LULC", str(row.get("to_lulc", "")).strip())
    entity_ids["CHANGE"] = add_entity("CHANGE", str(row.get("change_indicator", "")).strip())
    entity_ids["PROCESS"] = add_entity("PROCESS", str(row.get("lulc_process", "")).strip())
    entity_ids["MAGNITUDE"] = add_entity("MAGNITUDE", str(row.get("magnitude_percent", "")).strip() or str(row.get("magnitude_area", "")).strip())

    # Add relations if possible
    if entity_ids["CHANGE"] and entity_ids["PROCESS"]:
        results.append({
            "type": "relation",
            "from_id": entity_ids["CHANGE"],
            "to_id": entity_ids["PROCESS"],
            "labels": ["causes"],
            "direction": "right"
        })

    if entity_ids["PROCESS"] and entity_ids["TO_LULC"]:
        results.append({
            "type": "relation",
            "from_id": entity_ids["PROCESS"],
            "to_id": entity_ids["TO_LULC"],
            "labels": ["affects"],
            "direction": "right"
        })

    if entity_ids["FROM_LULC"] and entity_ids["TO_LULC"]:
        results.append({
            "type": "relation",
            "from_id": entity_ids["FROM_LULC"],
            "to_id": entity_ids["TO_LULC"],
            "labels": ["located_in"],
            "direction": "right"
        })

    if entity_ids["FROM_LULC"] and entity_ids["CHANGE"]:
        results.append({
            "type": "relation",
            "from_id": entity_ids["FROM_LULC"],
            "to_id": entity_ids["CHANGE"],
            "labels": ["temporal"],
            "direction": "right"
        })

    # Create the task for Label Studio
    if results:
        labelstudio_data.append({
            "id": idx + 1,
            "data": {"text": sentence},
            "annotations": [{"id": idx + 1000, "result": results}]
        })

# Save to JSON
with open("llama3_full_labelstudio.json", "w") as f:
    json.dump(labelstudio_data, f, indent=2)


In [ ]:
import pandas as pd
import json
import uuid
import os
import re # Import re for more complex cleaning

# --- CONFIGURATION ---
# IMPORTANT: Replace this with the actual path to your CSV file!
CSV_PATH = "llama3_first_10_examples.csv"
OUTPUT_JSON_PATH = "llama3_full_labelstudio.json"
# --- END CONFIGURATION ---

labelstudio_data = []

def get_first_span_case_insensitive(text_to_find, sentence_text):
    """Finds the first occurrence of text_to_find in sentence_text, case-insensitively."""
    if not text_to_find or pd.isna(text_to_find):
        return None, None
    
    text_to_find_str = str(text_to_find).strip()
    sentence_text_str = str(sentence_text)

    if not text_to_find_str: # If after stripping it's empty
        return None, None

    try:
        # Escape special regex characters in text_to_find for safe searching
        # This is important if text_to_find can contain characters like '.', '(', ')', '%'
        escaped_text_to_find = re.escape(text_to_find_str)
        match = re.search(escaped_text_to_find, sentence_text_str, re.IGNORECASE)
        if match:
            return match.start(), match.end()
        return None, None
    except ValueError: # Should not happen with re.search but good practice
        return None, None
    except Exception as e:
        print(f"Error in get_first_span_case_insensitive with '{text_to_find_str}': {e}")
        return None, None


def clean_entity_value_from_csv(raw_value, role_prefix_to_extract=None):
    """
    Cleans entity values from CSV.
    - Handles NaN, None, empty strings.
    - Removes common placeholder structures like "FROM: ..." or "MAGNITUDE:".
    - If role_prefix_to_extract (e.g., "FROM:") is given, it tries to extract text after it.
    Returns the cleaned entity text or an empty string if no valid entity is found.
    """
    if pd.isna(raw_value):
        return ""
    
    text = str(raw_value).strip()
    if not text or text.lower() == "nan":
        return ""

    # List of prefixes that indicate placeholder content or structure
    # Ensure they end with ":" to be more specific
    placeholder_markers = ["FROM:", "TO:", "CHANGE:", "PROCESS:", "MAGNITUDE:"]

    # If a specific role_prefix is provided (e.g., "FROM:"),
    # and the text starts with it (case-insensitive), try to extract the value after it.
    if role_prefix_to_extract:
        role_prefix_upper = role_prefix_to_extract.upper()
        if text.upper().startswith(role_prefix_upper):
            potential_value = text[len(role_prefix_upper):].strip()
            # Check if this extracted value itself is just another placeholder or empty
            if not potential_value:
                return ""
            
            # Further check: if this potential_value starts with another known marker,
            # it's likely part of a multi-line placeholder structure (e.g., "FROM: TO: X")
            # In this case, the value for "FROM:" is considered empty/not found.
            for marker in placeholder_markers:
                if potential_value.upper().startswith(marker.upper()):
                    return "" # It's a lead '{content_text_clean}' for role {role_label} not found in sentence.")
            pass
        return None

    from_lulc_text = row.get("from_lulc", "")
    to_lulc_text = row.get("to_lulc", "")
    change_indicator_text = row.get("change_indicator", "")
    lulc_process_text = row.get("lulc_process", "")
    magnitude_percent_text = row.get("magnitude_percent", "")
    magnitude_area_text = row.get("magnitude_area", "")

    magnitude_text_to_use = str(magnitude_percent_text).strip()
    if not magnitude_text_to_use or magnitude_text_to_use.lower() == 'nan' or \
       any(magnitude_text_to_use.lower().startswith(p) for p in LLM_PLACEHOLDER_PREFIXES):
        magnitude_text_to_use = str(magnitude_area_text).strip()
        if not magnitude_text_to_use or magnitude_text_to_use.lower() == 'nan' or \
           any(magnitude_text_to_use.lower().startswith(p) for p in LLM_PLACEHOLDER_PREFIXES):
            magnitude_text_to_use = ""


    found_entity_eids_by_role["FROM_LULC"] = add_entity_to_task("FROM_LULC", from_lulc_text)
    found_entity_eids_by_role["TO_LULC"] = add_entity_to_task("TO_LULC", to_lulc_text)
    found_entity_eids_by_role["CHANGE"] = add_entity_to_task("CHANGE", change_indicator_text)
    found_entity_eids_by_role["PROCESS"] = add_entity_to_task("PROCESS", lulc_process_text)
    found_entity_eids_by_role["MAGNITUDE"] = add_entity_to_task("MAGNITUDE", magnitude_text_to_use)

    # --- Add relations (Adjust relation labels as needed) ---
    # Relation: FROM_LULC converts_to TO_LULC
    if found_entity_eids_by_role.get("FROM_LULC") and found_entity_eids_by_role.get("TO_LULC"):
        # Avoid self-relation if FROM and TO are the same entity instance and same text/column source
        # A more robust check might be needed if "built-up area" is FROM and "built-up area" is TO but they
        # are meant to be distinct points in a process. For now, basic check.
        if found_entity_eids_by_role["FROM_LULC"] != found_entity_eids_by_role["TO_LULC"] or \
           str(from_lulc_text).strip().lower() != str(to_lulc_text).strip().lower():
            current_task_results.append({
                "id": str(uuid.uuid4()), "from_name": "relation", "to_name": "text", "type": "relation",
                "value": {"from_id": found_entity_eids_by_role["FROM_LULC"], "to_id": found_entity_eids_by_role["TO_LULC"], "labels": ["converts_to"], "direction": "right"}
            })

    # Relation: CHANGE is_part_of PROCESS (or PROCESS includes CHANGE)
    if found_entity_eids_by_role.get("CHANGE") and found_entity_eids_by_role.get("PROCESS"):
        current_task_results.append({
            "id": str(uuid.uuid4()), "from_name": "relation", "to_name": "text", "type": "relation",
            "value": {"from_id": found_entity_eids_by_role["CHANGE"], "to_id": found_entity_eids_by_role["PROCESS"], "labels": ["part_of"], "direction": "right"}
        })

    # Relation: CHANGE has_magnitude MAGNITUDE
    if found_entity_eids_by_role.get("CHANGE") and found_entity_eids_by_role.get("MAGNITUDE"):
        current_task_results.append({
            "id": str(uuid.uuid4()), "from_name": "relation", "to_name": "text", "type": "relation",
            "value": {"from_id": found_entity_eids_by_role["CHANGE"], "to_id": found_entity_eids_by_role["MAGNITUDE"], "labels": ["has_magnitude"], "direction": "right"}
        })
    
    # You can add more relations based on your specific schema, for example:
    # - FROM_LULC -> CHANGE (e.g., "undergoes")
    # - PROCESS -> TO_LULC (e.g., "results_in")


    if current_task_results: # Only add if there's at least one label or relation
        labelstudio_data.append({
            "id": idx + 1, # Task ID for Label Studio
            "data": {"text": original_sentence},
            "annotations": [{"result": current_task_results}]
        })
        tasks_generated_count += 1
    # else:
        # print(f"Row {idx+2}: No annotations generated for this row.")

print(f"\nFinished iteration. Processed {idx + 1 if 'idx' in locals() else 0} data rows.")
print(f"Number of tasks created with annotations for Label Studio: {tasks_generated_count}")

if labelstudio_data:
    with open(OUTPUT_JSON_PATH, "w") as f:
        json.dump(labelstudio_data, f, indent=2)
    print(f"Successfully saved {len(labelstudio_data)} tasks to {OUTPUT_JSON_PATH}")
else:
    print("No data was processed to save to JSON. Output file will be empty or not created.")

In [18]:
# Load the saved CSV file
df = pd.read_csv("llama3_first_10_examples.csv")

# Create the Label Studio data structure
labelstudio_data = []

def get_span(text, sentence):
    try:
        start = sentence.index(text)
        return start, start + len(text)
    except ValueError:
        return None, None

for idx, row in df.iterrows():
    sentence = str(row["original_sentence"])
    results = []
    entities = {}

    def add_entity(label, content):
        if isinstance(content, str) and content and content.lower() != "nan" and content in sentence:
            start, end = get_span(content, sentence)
            if start is not None:
                eid = str(uuid.uuid4())
                results.append({
                    "id": eid,
                    "from_name": "label",
                    "to_name": "text",
                    "type": "labels",
                    "value": {"start": start, "end": end, "text": content, "labels": [label]}
                })
                return eid
        return None

    # Extract all entities
    entities['FROM_LULC'] = add_entity("FROM_LULC", str(row.get("from_lulc", "")).strip())
    entities['TO_LULC'] = add_entity("TO_LULC", str(row.get("to_lulc", "")).strip())
    entities['CHANGE'] = add_entity("CHANGE", str(row.get("change_indicator", "")).strip())
    entities['PROCESS'] = add_entity("PROCESS", str(row.get("lulc_process", "")).strip())
    entities['MAGNITUDE'] = add_entity("MAGNITUDE", str(row.get("magnitude_percent", "")).strip() or str(row.get("magnitude_area", "")).strip())

    # Add sample relations if valid
    if entities["CHANGE"] and entities["PROCESS"]:
        results.append({
            "type": "relation",
            "from_id": entities["CHANGE"],
            "to_id": entities["PROCESS"],
            "labels": ["causes"],
            "direction": "right"
        })
    if entities["PROCESS"] and entities["TO_LULC"]:
        results.append({
            "type": "relation",
            "from_id": entities["PROCESS"],
            "to_id": entities["TO_LULC"],
            "labels": ["affects"],
            "direction": "right"
        })
    if entities["FROM_LULC"] and entities["TO_LULC"]:
        results.append({
            "type": "relation",
            "from_id": entities["FROM_LULC"],
            "to_id": entities["TO_LULC"],
            "labels": ["located_in"],
            "direction": "right"
        })
    if entities["FROM_LULC"] and entities["CHANGE"]:
        results.append({
            "type": "relation",
            "from_id": entities["FROM_LULC"],
            "to_id": entities["CHANGE"],
            "labels": ["temporal"],
            "direction": "right"
        })

    if results:
        labelstudio_data.append({
            "id": idx + 1,
            "data": {"text": sentence},
            "annotations": [{"id": idx + 1000, "result": results}]
        })

# Save final JSON
output_path = "llama3_full_labelstudio.json"
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(labelstudio_data, f, indent=2)

output_path


'llama3_full_labelstudio.json'

In [20]:

# Filter the DataFrame to retain only rows where the extracted terms can be matched in the sentence
df = pd.read_csv("llama3_first_10_examples.csv")

def is_matchable(term, sentence):
    return isinstance(term, str) and term.strip() and term.strip().lower() != "nan" and term.strip() in sentence

cleaned_df = df[
    df.apply(
        lambda row: any([
            is_matchable(row.get("from_lulc"), row["original_sentence"]),
            is_matchable(row.get("to_lulc"), row["original_sentence"]),
            is_matchable(row.get("change_indicator"), row["original_sentence"]),
            is_matchable(row.get("lulc_process"), row["original_sentence"]),
            is_matchable(row.get("magnitude_percent"), row["original_sentence"]),
            is_matchable(row.get("magnitude_area"), row["original_sentence"])
        ]),
        axis=1
    )
]

# Save the cleaned DataFrame
cleaned_csv_path = "cleaned_lulc_events.csv"
cleaned_df.to_csv(cleaned_csv_path, index=False)

cleaned_csv_path


'cleaned_lulc_events.csv'

In [22]:
import pandas as pd
import json
import uuid

# Load the cleaned CSV
df = pd.read_csv("llama3_first_10_examples.csv")

# Output structure for Label Studio
labelstudio_data = []

def find_span(entity, sentence):
    entity = str(entity).strip().lower()
    sentence_lower = sentence.lower()
    try:
        start = sentence_lower.index(entity)
        return start, start + len(entity)
    except ValueError:
        # Try partial matching as fallback
        for word in entity.split():
            if len(word) > 3 and word in sentence_lower:  # Only match significant words
                start = sentence_lower.index(word)
                return start, start + len(word)
        return None, None

for idx, row in df.iterrows():
    sentence = str(row["original_sentence"])
    results = []
    entities = {}

    def add_entity(label_name, entity_text):
        if isinstance(entity_text, str) and entity_text.strip() and entity_text in sentence:
            start, end = find_span(entity_text, sentence)
            if start is not None:
                eid = str(uuid.uuid4())
                results.append({
                    "id": eid,
                    "from_name": "label",
                    "to_name": "text",
                    "type": "labels",
                    "value": {"start": start, "end": end, "text": entity_text, "labels": [label_name]}
                })
                return eid
        return None

    # Add entities
    entities["FROM_LULC"] = add_entity("FROM_LULC", str(row.get("from_lulc", "")).strip())
    entities["TO_LULC"] = add_entity("TO_LULC", str(row.get("to_lulc", "")).strip())
    entities["CHANGE"] = add_entity("CHANGE", str(row.get("change_indicator", "")).strip())
    entities["PROCESS"] = add_entity("PROCESS", str(row.get("lulc_process", "")).strip())
    entities["MAGNITUDE"] = add_entity("MAGNITUDE", str(row.get("magnitude_percent", "")).strip() or str(row.get("magnitude_area", "")).strip())

    # Add relations between valid pairs
    if entities["CHANGE"] and entities["PROCESS"]:
        results.append({
            "type": "relation",
            "from_id": entities["CHANGE"],
            "to_id": entities["PROCESS"],
            "labels": ["causes"],
            "direction": "right"
        })
    if entities["PROCESS"] and entities["TO_LULC"]:
        results.append({
            "type": "relation",
            "from_id": entities["PROCESS"],
            "to_id": entities["TO_LULC"],
            "labels": ["affects"],
            "direction": "right"
        })
    if entities["FROM_LULC"] and entities["TO_LULC"]:
        results.append({
            "type": "relation",
            "from_id": entities["FROM_LULC"],
            "to_id": entities["TO_LULC"],
            "labels": ["located_in"],
            "direction": "right"
        })
    if entities["FROM_LULC"] and entities["CHANGE"]:
        results.append({
            "type": "relation",
            "from_id": entities["FROM_LULC"],
            "to_id": entities["CHANGE"],
            "labels": ["temporal"],
            "direction": "right"
        })

    if results:
        labelstudio_data.append({
            "id": idx + 1,
            "data": {"text": sentence},
            "annotations": [{"id": idx + 1000, "result": results}]
        })
# Add at the beginning of your loop
print(f"Processing row {idx}: {sentence[:50]}...")
# Add after processing
print(f"Found {len(results)} entities/relations for row {idx}")

# Save JSON
final_output_path = "labelstudio_relation_extraction.json"
with open(final_output_path, "w", encoding="utf-8") as f:
    json.dump(labelstudio_data, f, indent=2)

final_output_path


Processing row 9: The resultant improvement in basic facilities comb...
Found 0 entities/relations for row 9


'labelstudio_relation_extraction.json'

In [24]:
import pandas as pd
import json
import uuid
import re

# Load the cleaned CSV
df = pd.read_csv("llama3_first_10_examples.csv")

# Output structure for Label Studio
labelstudio_data = []

# Improved span finding function
def find_span(entity, sentence):
    if not isinstance(entity, str) or not entity.strip():
        return None, None
    
    entity = entity.strip().lower()
    sentence_lower = sentence.lower()
    
    # Try exact matching first
    try:
        start = sentence_lower.index(entity)
        return start, start + len(entity)
    except ValueError:
        # Try more flexible matching
        # 1. Try without punctuation
        clean_entity = re.sub(r'[^\w\s]', '', entity)
        clean_sentence = re.sub(r'[^\w\s]', '', sentence_lower)
        try:
            start = clean_sentence.index(clean_entity)
            # Map back to original position (approximate)
            original_start = 0
            for i, char in enumerate(clean_sentence[:start]):
                while original_start < len(sentence_lower) and (sentence_lower[original_start].isspace() or not sentence_lower[original_start].isalnum()):
                    original_start += 1
                if i < start:
                    original_start += 1
            
            # Approximate end position
            original_end = original_start
            for i in range(len(clean_entity)):
                while original_end < len(sentence_lower) and (sentence_lower[original_end].isspace() or not sentence_lower[original_end].isalnum()):
                    original_end += 1
                original_end += 1
            
            return original_start, original_end
        except ValueError:
            # 2. Try word-by-word matching for multi-word entities
            if len(entity.split()) > 1:
                for word in entity.split():
                    if len(word) > 3 and word in sentence_lower:  # Only match significant words
                        start = sentence_lower.index(word)
                        return start, start + len(word)
            return None, None

print(f"Total rows in CSV: {len(df)}")
print("First few rows:")
print(df[['original_sentence', 'from_lulc', 'to_lulc', 'change_indicator', 'lulc_process']].head())

if 'event_found' in df.columns:
    print("Event counts:")
    print(df['event_found'].value_counts())

for idx, row in df.iterrows():
    sentence = str(row["original_sentence"])
    print(f"Processing row {idx}: {sentence[:50]}...")
    results = []
    entities = {}

    def add_entity(label_name, entity_text):
        if pd.isna(entity_text):
            return None
            
        entity_text = str(entity_text).strip()
        if not entity_text:
            return None
            
        start, end = find_span(entity_text, sentence)
        if start is not None:
            eid = str(uuid.uuid4())
            results.append({
                "id": eid,
                "from_name": "label",
                "to_name": "text",
                "type": "labels",
                "value": {"start": start, "end": end, "text": sentence[start:end], "labels": [label_name]}
            })
            print(f"  Found {label_name}: '{entity_text}' at position {start}-{end}")
            return eid
        else:
            print(f"  Could not find {label_name}: '{entity_text}' in text")
        return None

    # Add entities
    entities["FROM_LULC"] = add_entity("FROM_LULC", row.get("from_lulc", ""))
    entities["TO_LULC"] = add_entity("TO_LULC", row.get("to_lulc", ""))
    entities["CHANGE"] = add_entity("CHANGE", row.get("change_indicator", ""))
    entities["PROCESS"] = add_entity("PROCESS", row.get("lulc_process", ""))
    
    # Handle magnitude from either percent or area
    magnitude_percent = row.get("magnitude_percent", "")
    magnitude_area = row.get("magnitude_area", "")
    magnitude = magnitude_area if (isinstance(magnitude_area, str) and magnitude_area.strip()) else magnitude_percent
    entities["MAGNITUDE"] = add_entity("MAGNITUDE", magnitude)

    # Add relations between valid pairs
    relation_count = 0
    if entities["FROM_LULC"] and entities["CHANGE"]:
        results.append({
            "id": f"relation-{idx}-{relation_count}",
            "from_name": "relation",
            "to_name": "text",
            "type": "relation",
            "value": {
                "from_id": entities["FROM_LULC"],
                "to_id": entities["CHANGE"],
                "labels": ["causes"],
                "direction": "right"
            }
        })
        relation_count += 1
        print(f"  Added relation: FROM_LULC causes CHANGE")
        
    if entities["CHANGE"] and entities["PROCESS"]:
        results.append({
            "id": f"relation-{idx}-{relation_count}",
            "from_name": "relation",
            "to_name": "text",
            "type": "relation",
            "value": {
                "from_id": entities["CHANGE"],
                "to_id": entities["PROCESS"],
                "labels": ["part_of"],
                "direction": "right"
            }
        })
        relation_count += 1
        print(f"  Added relation: CHANGE part_of PROCESS")
        
    if entities["FROM_LULC"] and entities["TO_LULC"]:
        results.append({
            "id": f"relation-{idx}-{relation_count}",
            "from_name": "relation",
            "to_name": "text",
            "type": "relation",
            "value": {
                "from_id": entities["FROM_LULC"],
                "to_id": entities["TO_LULC"],
                "labels": ["converts_to"],
                "direction": "right"
            }
        })
        relation_count += 1
        print(f"  Added relation: FROM_LULC converts_to TO_LULC")
        
    if entities["CHANGE"] and entities["MAGNITUDE"]:
        results.append({
            "id": f"relation-{idx}-{relation_count}",
            "from_name": "relation",
            "to_name": "text",
            "type": "relation",
            "value": {
                "from_id": entities["CHANGE"],
                "to_id": entities["MAGNITUDE"],
                "labels": ["affects"],
                "direction": "right"
            }
        })
        relation_count += 1
        print(f"  Added relation: CHANGE affects MAGNITUDE")

    print(f"Found {len(results)} entities/relations for row {idx}")
    
    if results:
        labelstudio_data.append({
            "id": idx + 1,
            "data": {"text": sentence},
            "annotations": [{"id": idx + 1000, "result": results}]
        })

print(f"Total items added to Label Studio JSON: {len(labelstudio_data)}")

# Save JSON
final_output_path = "labelstudio_relation_extraction.json"
with open(final_output_path, "w", encoding="utf-8") as f:
    json.dump(labelstudio_data, f, indent=2)

print(f"Saved Label Studio JSON to {final_output_path}")


Total rows in CSV: 10
First few rows:
                                   original_sentence  \
0  Simulation results reveal that the landscape o...   
1  The study observed a significant increase (12....   
2  On the contrary, forest cover declined drastic...   
3  Rapid population growth triggered by rural urb...   
4  Under the business as usual scenario, predicti...   

                               from_lulc                          to_lulc  \
0  TO: \nCHANGE: \nPROCESS: \nMAGNITUDE:  CHANGE: \nPROCESS: \nMAGNITUDE:   
1                          built-up area                    built-up area   
2  TO: \nCHANGE: \nPROCESS: \nMAGNITUDE:  CHANGE: \nPROCESS: \nMAGNITUDE:   
3                      agricultural land                      urban areas   
4                          built-up area                    built-up area   

        change_indicator  lulc_process  
0  PROCESS: \nMAGNITUDE:    MAGNITUDE:  
1              increased  urbanization  
2  PROCESS: \nMAGNITUDE:    MAGNITUDE: 

In [25]:
# Redefine with fixed variable scope (no `nonlocal` required outside nested function)
import pandas as pd
import json
import uuid
import re

# Load the cleaned CSV
df = pd.read_csv("llama3_first_10_examples.csv")

# Helper function: clean and normalize entity text
def normalize(text):
    if not isinstance(text, str):
        return ""
    return re.sub(r"\s+", " ", text).strip().lower()

# Helper function: find entity span in a sentence using flexible matching
def find_entity_span(entity, sentence):
    if not entity or not isinstance(entity, str):
        return None, None
    entity_clean = normalize(entity)
    sentence_clean = normalize(sentence)

    start_idx = sentence_clean.find(entity_clean)
    if start_idx == -1:
        return None, None

    # Use regex in original sentence to get actual span
    pattern = re.compile(re.escape(entity), re.IGNORECASE)
    match = pattern.search(sentence)
    if match:
        return match.start(), match.end()
    return None, None

# Statistics counters
stats = {
    "total_rows": len(df),
    "matched_rows": 0,
    "total_entities": 0,
    "matched_entities": 0
}

labelstudio_data = []

for idx, row in df.iterrows():
    sentence = str(row["original_sentence"])
    results = []
    entities = {}

    def add_entity(label_name, entity_text):
        stats["total_entities"] += 1
        entity_text = str(entity_text).strip()
        start, end = find_entity_span(entity_text, sentence)
        if start is not None:
            eid = str(uuid.uuid4())
            results.append({
                "id": eid,
                "from_name": "label",
                "to_name": "text",
                "type": "labels",
                "value": {
                    "start": start,
                    "end": end,
                    "text": sentence[start:end],
                    "labels": [label_name]
                }
            })
            stats["matched_entities"] += 1
            return eid
        return None

    # Add all entity types
    entities["FROM_LULC"] = add_entity("FROM_LULC", row.get("from_lulc", ""))
    entities["TO_LULC"] = add_entity("TO_LULC", row.get("to_lulc", ""))
    entities["CHANGE"] = add_entity("CHANGE", row.get("change_indicator", ""))
    entities["PROCESS"] = add_entity("PROCESS", row.get("lulc_process", ""))
    entities["MAGNITUDE"] = add_entity("MAGNITUDE", row.get("magnitude_percent", "") or row.get("magnitude_area", ""))

    # Add valid relations
    def add_relation(label, from_key, to_key):
        if entities[from_key] and entities[to_key]:
            results.append({
                "type": "relation",
                "from_id": entities[from_key],
                "to_id": entities[to_key],
                "labels": [label],
                "direction": "right"
            })

    add_relation("causes", "CHANGE", "PROCESS")
    add_relation("affects", "PROCESS", "TO_LULC")
    add_relation("located_in", "FROM_LULC", "TO_LULC")
    add_relation("temporal", "FROM_LULC", "CHANGE")

    if results:
        stats["matched_rows"] += 1
        labelstudio_data.append({
            "id": idx + 1,
            "data": {"text": sentence},
            "annotations": [{"id": idx + 1000, "result": results}]
        })

# Save output
output_path = "labelstudio_relation_extraction.json"
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(labelstudio_data, f, indent=2)

# Return statistics and path
stats["output_path"] = output_path
stats


{'total_rows': 10,
 'matched_rows': 1,
 'total_entities': 50,
 'matched_entities': 3,
 'output_path': 'labelstudio_relation_extraction.json'}

In [26]:
# Load the auto-cleaned CSV
df_cleaned = pd.read_csv("llama3_first_10_examples.csv")

# Reset stats and results
labelstudio_data = []
stats = {
    "total_rows": len(df_cleaned),
    "matched_rows": 0,
    "total_entities": 0,
    "matched_entities": 0
}

# Flexible matching helpers
def normalize(text):
    if not isinstance(text, str):
        return ""
    return re.sub(r"\s+", " ", text).strip().lower()

def find_entity_span(entity, sentence):
    if not entity or not isinstance(entity, str):
        return None, None
    entity_clean = normalize(entity)
    sentence_clean = normalize(sentence)
    start_idx = sentence_clean.find(entity_clean)
    if start_idx == -1:
        return None, None
    pattern = re.compile(re.escape(entity), re.IGNORECASE)
    match = pattern.search(sentence)
    if match:
        return match.start(), match.end()
    return None, None

for idx, row in df_cleaned.iterrows():
    sentence = str(row["original_sentence"])
    results = []
    entities = {}

    def add_entity(label_name, entity_text):
        stats["total_entities"] += 1
        entity_text = str(entity_text).strip()
        start, end = find_entity_span(entity_text, sentence)
        if start is not None:
            eid = str(uuid.uuid4())
            results.append({
                "id": eid,
                "from_name": "label",
                "to_name": "text",
                "type": "labels",
                "value": {
                    "start": start,
                    "end": end,
                    "text": sentence[start:end],
                    "labels": [label_name]
                }
            })
            stats["matched_entities"] += 1
            return eid
        return None

    # Add entities
    entities["FROM_LULC"] = add_entity("FROM_LULC", row.get("from_lulc", ""))
    entities["TO_LULC"] = add_entity("TO_LULC", row.get("to_lulc", ""))
    entities["CHANGE"] = add_entity("CHANGE", row.get("change_indicator", ""))
    entities["PROCESS"] = add_entity("PROCESS", row.get("lulc_process", ""))
    entities["MAGNITUDE"] = add_entity("MAGNITUDE", row.get("magnitude_percent", "") or row.get("magnitude_area", ""))

    # Add valid relations
    def add_relation(label, from_key, to_key):
        if entities[from_key] and entities[to_key]:
            results.append({
                "type": "relation",
                "from_id": entities[from_key],
                "to_id": entities[to_key],
                "labels": [label],
                "direction": "right"
            })

    add_relation("causes", "CHANGE", "PROCESS")
    add_relation("affects", "PROCESS", "TO_LULC")
    add_relation("located_in", "FROM_LULC", "TO_LULC")
    add_relation("temporal", "FROM_LULC", "CHANGE")

    if results:
        stats["matched_rows"] += 1
        labelstudio_data.append({
            "id": idx + 1,
            "data": {"text": sentence},
            "annotations": [{"id": idx + 1000, "result": results}]
        })

# Save output
final_output = "labelstudio_from_autocleaned.json"
with open(final_output, "w", encoding="utf-8") as f:
    json.dump(labelstudio_data, f, indent=2)

# Include stats
stats["output_path"] = final_output
stats


{'total_rows': 10,
 'matched_rows': 1,
 'total_entities': 50,
 'matched_entities': 3,
 'output_path': 'labelstudio_from_autocleaned.json'}

In [27]:
import pandas as pd
import json
import uuid
import re

# Load the user's CSV file
csv_path = "llama3_first_10_examples.csv"
df = pd.read_csv(csv_path)

labelstudio_data = []

def normalize(text):
    return re.sub(r'\s+', ' ', str(text)).strip().lower()

def find_span(entity, sentence):
    if not isinstance(entity, str) or not entity.strip():
        return None, None
    entity = normalize(entity)
    sentence_lower = sentence.lower()
    try:
        start = sentence_lower.index(entity)
        return start, start + len(entity)
    except ValueError:
        return None, None

for idx, row in df.iterrows():
    sentence = str(row["original_sentence"])
    results = []
    entities = {}

    def add_entity(label, value):
        if pd.isna(value) or not str(value).strip():
            return None
        value_str = str(value).strip()
        start, end = find_span(value_str, sentence)
        if start is not None:
            eid = str(uuid.uuid4())
            results.append({
                "id": eid,
                "from_name": "label",
                "to_name": "text",
                "type": "labels",
                "value": {
                    "start": start,
                    "end": end,
                    "text": sentence[start:end],
                    "labels": [label]
                }
            })
            return eid
        return None

    # Add labeled entities
    entities["FROM_LULC"] = add_entity("FROM_LULC", row.get("from_lulc", ""))
    entities["TO_LULC"] = add_entity("TO_LULC", row.get("to_lulc", ""))
    entities["CHANGE"] = add_entity("CHANGE", row.get("change_indicator", ""))
    entities["PROCESS"] = add_entity("PROCESS", row.get("lulc_process", ""))
    magnitude = row.get("magnitude_area", "") or row.get("magnitude_percent", "")
    entities["MAGNITUDE"] = add_entity("MAGNITUDE", magnitude)

    # Add relations
    def add_relation(label, source, target):
        if entities[source] and entities[target]:
            results.append({
                "type": "relation",
                "from_id": entities[source],
                "to_id": entities[target],
                "labels": [label]
            })

    add_relation("causes", "FROM_LULC", "CHANGE")
    add_relation("part_of", "CHANGE", "PROCESS")
    add_relation("converts_to", "FROM_LULC", "TO_LULC")
    add_relation("affects", "CHANGE", "MAGNITUDE")

    if results:
        labelstudio_data.append({
            "id": idx + 1,
            "data": {"text": sentence},
            "annotations": [{
                "id": idx + 1000,
                "result": results
            }]
        })

# Save corrected Label Studio JSON format
fixed_output_path = "fixed_labelstudio_format.json"
with open(fixed_output_path, "w", encoding="utf-8") as f:
    json.dump(labelstudio_data, f, indent=2)

fixed_output_path


'fixed_labelstudio_format.json'

In [28]:
import json
import uuid

# Sample input
sentence = "The study observed a significant increase (12.77%) in built-up area from 2002 (52.88%) to 2018 (65.5%), followed by a slight increase in the cover of bare ground."
from_lulc = "built-up area"
to_lulc = "built-up area"
change = "increased"
process = "urbanization"
magnitude = "12.77%"

# Find spans
def find_span(text, sentence):
    start = sentence.lower().find(text.lower())
    if start == -1:
        return None, None
    return start, start + len(text)

# Build entities
results = []
entity_ids = {}

def add_entity(label, value):
    start, end = find_span(value, sentence)
    if start is not None:
        eid = str(uuid.uuid4())
        results.append({
            "id": eid,
            "from_name": "label",
            "to_name": "text",
            "type": "labels",
            "value": {
                "start": start,
                "end": end,
                "text": sentence[start:end],
                "labels": [label]
            }
        })
        entity_ids[label] = eid

# Add each entity
add_entity("FROM_LULC", from_lulc)
add_entity("TO_LULC", to_lulc)
add_entity("CHANGE", change)
add_entity("PROCESS", process)
add_entity("MAGNITUDE", magnitude)

# Add relations
def add_relation(from_label, to_label, relation_label):
    if from_label in entity_ids and to_label in entity_ids:
        results.append({
            "type": "relation",
            "from_id": entity_ids[from_label],
            "to_id": entity_ids[to_label],
            "labels": [relation_label]
        })

add_relation("FROM_LULC", "CHANGE", "causes")
add_relation("CHANGE", "PROCESS", "part_of")
add_relation("FROM_LULC", "TO_LULC", "converts_to")
add_relation("CHANGE", "MAGNITUDE", "affects")

# Build the full task
labelstudio_task = [{
    "id": 1,
    "data": {"text": sentence},
    "annotations": [{
        "id": 1001,
        "result": results
    }]
}]

# Save to file
single_output_path = "labelstudio_single_relation.json"
with open(single_output_path, "w", encoding="utf-8") as f:
    json.dump(labelstudio_task, f, indent=2)

single_output_path


'labelstudio_single_relation.json'

In [29]:
# New example sentence (with missing structured field values)
sentence2 = "On the contrary, forest cover declined drastically (15.25%) followed by agriculture (1.01%)."
from_lulc2 = "forest cover"
to_lulc2 = "agriculture"
change2 = "declined"
process2 = "deforestation"
magnitude2 = "15.25%"

# Create fresh results list
results2 = []
entity_ids2 = {}

# Entity adder
def add_entity2(label, value):
    start, end = find_span(value, sentence2)
    if start is not None:
        eid = str(uuid.uuid4())
        results2.append({
            "id": eid,
            "from_name": "label",
            "to_name": "text",
            "type": "labels",
            "value": {
                "start": start,
                "end": end,
                "text": sentence2[start:end],
                "labels": [label]
            }
        })
        entity_ids2[label] = eid

# Add all entities
add_entity2("FROM_LULC", from_lulc2)
add_entity2("TO_LULC", to_lulc2)
add_entity2("CHANGE", change2)
add_entity2("PROCESS", process2)
add_entity2("MAGNITUDE", magnitude2)

# Add relations
def add_relation2(from_label, to_label, relation_label):
    if from_label in entity_ids2 and to_label in entity_ids2:
        results2.append({
            "type": "relation",
            "from_id": entity_ids2[from_label],
            "to_id": entity_ids2[to_label],
            "labels": [relation_label]
        })

add_relation2("FROM_LULC", "CHANGE", "causes")
add_relation2("CHANGE", "PROCESS", "part_of")
add_relation2("FROM_LULC", "TO_LULC", "converts_to")
add_relation2("CHANGE", "MAGNITUDE", "affects")

# Wrap task
labelstudio_task2 = [{
    "id": 2,
    "data": {"text": sentence2},
    "annotations": [{
        "id": 2002,
        "result": results2
    }]
}]

# Save
output_path2 = "labelstudio_single_relation_forest.json"
with open(output_path2, "w", encoding="utf-8") as f:
    json.dump(labelstudio_task2, f, indent=2)

output_path2


'labelstudio_single_relation_forest.json'

In [31]:
import json
import uuid
import re

# Input sentence and structured info
sentence = (
    "Under the business as usual scenario, prediction analysis for the year 2050 show that built up area will consume almost all of the city area (73.21%) with forest significantly reduced to patches making up only about 16% of the city."
)
from_lulc = "built-up area"
to_lulc = "built-up area"
change = "increased"
process = "urbanization"
magnitude = "12.77%"

# Helper function to find fuzzy span
def find_span_fuzzy(entity, sentence):
    if not isinstance(entity, str) or not entity.strip():
        return None, None
    entity_clean = re.sub(r"[^\w\s]", "", entity.lower())
    sentence_clean = re.sub(r"[^\w\s]", "", sentence.lower())
    idx = sentence_clean.find(entity_clean)
    if idx == -1:
        return None, None

    # Map back to original (approximate)
    words_original = sentence.split()
    char_pos = 0
    for word in words_original:
        if entity_clean.startswith(re.sub(r"[^\w\s]", "", word.lower())):
            approx_start = sentence.lower().find(word.lower(), char_pos)
            if approx_start != -1:
                return approx_start, approx_start + len(entity)
        char_pos += len(word) + 1
    return None, None

# Entity + relation tracking
results = []
entity_ids = {}

def add_entity(label, value):
    start, end = find_span_fuzzy(value, sentence)
    if start is not None:
        eid = str(uuid.uuid4())
        results.append({
            "id": eid,
            "from_name": "label",
            "to_name": "text",
            "type": "labels",
            "value": {
                "start": start,
                "end": end,
                "text": sentence[start:end],
                "labels": [label]
            }
        })
        entity_ids[label] = eid

# Add all entities
add_entity("FROM_LULC", from_lulc)
add_entity("TO_LULC", to_lulc)
add_entity("CHANGE", change)
add_entity("PROCESS", process)
add_entity("MAGNITUDE", magnitude)

# Add relation edges
def add_relation(from_label, to_label, rel_label):
    if from_label in entity_ids and to_label in entity_ids:
        results.append({
            "type": "relation",
            "from_id": entity_ids[from_label],
            "to_id": entity_ids[to_label],
            "labels": [rel_label]
        })

add_relation("FROM_LULC", "CHANGE", "causes")
add_relation("CHANGE", "PROCESS", "part_of")
add_relation("FROM_LULC", "TO_LULC", "converts_to")
add_relation("CHANGE", "MAGNITUDE", "affects")

# Final Label Studio task
labelstudio_task = [{
    "id": 1,
    "data": {"text": sentence},
    "annotations": [{
        "id": 1001,
        "result": results
    }]
}]

# Save to file
output_path = "labelstudio_single_relation_builtup.json"
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(labelstudio_task, f, indent=2)

print(f"✅ Saved corrected Label Studio file to: {output_path}")


✅ Saved corrected Label Studio file to: labelstudio_single_relation_builtup.json
